## 1. Tooth Detection and Segmentation
Process dental cases with the Tooth Segmentation + Recognition model.
This step outputs bounding-box images, mask overlays, and per-case JSON files.

Next step output used by Step 2:
- segmentation+recognition-dataset

In [ ]:
# Step 1: Tooth Detection and Segmentation on raw_data (all cases)
import os
import json
import random
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# IMPORTANT for Windows/Jupyter: use non-interactive backend to avoid Tkinter issues
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ultralytics import YOLO
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    union = float(areaA + areaB - inter)
    return inter / union if union > 0 else 0.0


def get_device():
    import torch

    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using device: cuda | GPU: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        print("Using device: mps")
    else:
        device = "cpu"
        print("Using device: cpu")
    return device


def get_detectron2_device(device: str) -> str:
    # Detectron2 reliably supports cuda/cpu. If main device is mps, use cpu for Detectron2.
    if device == "cuda":
        return "cuda"
    if device == "mps":
        print("Detectron2 uses cpu because mps support is limited.")
        return "cpu"
    return "cpu"


def initialize_models(raw_data_dir: Path):
    device = get_device()
    detectron_device = get_detectron2_device(device)

    # Try common locations for model weights (supports both 'weights' and 'weights(model)')
    yolo_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_pano_20250319.pt",
    ]
    detectron_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_crop_20250424.pth",
    ]

    yolo_path = next((p for p in yolo_candidates if p.exists()), None)
    detectron_path = next((p for p in detectron_candidates if p.exists()), None)

    if yolo_path is None or detectron_path is None:
        checked = [str(p) for p in (yolo_candidates + detectron_candidates)]
        raise FileNotFoundError(
            "Model weights not found. Checked paths:\n" + "\n".join(checked)
        )

    print(f"YOLO weights: {yolo_path}")
    print(f"Detectron2 weights: {detectron_path}")

    yolo_model = YOLO(str(yolo_path))
    yolo_model.to(device)

    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.WEIGHTS = str(detectron_path)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.DEVICE = detectron_device
    detectron_predictor = DefaultPredictor(cfg)

    print("Models initialized successfully")
    return yolo_model, detectron_predictor


def process_panoramic_image(pano_img_rgb, yolo_model):
    results = yolo_model(pano_img_rgb, verbose=False)
    h, w = pano_img_rgb.shape[:2]
    all_masks = []

    for result in results:
        if result.masks is None or not hasattr(result.masks, "data"):
            continue

        temp = []
        num_masks = len(result.masks.xy) if hasattr(result.masks, "xy") else 0

        for i in range(num_masks):
            mask_matrix = result.masks.data[i].cpu().numpy()
            resized_mask = cv2.resize(
                mask_matrix.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
            ).astype(bool)

            bbox = result.boxes.xyxy[i].cpu().numpy().tolist()
            cls_id = int(result.boxes.cls[i].item())

            temp.append({
                "id": i,
                "matrix_resized": resized_mask,
                "confidence": float(result.boxes.conf[i].item()),
                "class_id": cls_id,
                "class_name": str(result.names[cls_id]),
                "bbox": bbox,
            })

        # Remove duplicates by IoU
        kept = []
        for m in temp:
            duplicated = False
            for k in kept:
                if compute_iou(m["bbox"], k["bbox"]) > 0.5:
                    if m["confidence"] > k["confidence"]:
                        kept.remove(k)
                        kept.append(m)
                    duplicated = True
                    break
            if not duplicated:
                kept.append(m)

        all_masks.extend(kept)

    return all_masks


def crop_and_segment_teeth(pano_img_rgb, mask_data_list, detectron_predictor, pad=20):
    outputs = []
    for data in mask_data_list:
        mask = data["matrix_resized"].astype(np.uint8) * 255
        x, y, w, h = cv2.boundingRect(mask)
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, pano_img_rgb.shape[1])
        y2 = min(y + h + pad, pano_img_rgb.shape[0])

        cropped = pano_img_rgb[y1:y2, x1:x2]
        pred = detectron_predictor(cropped)
        instances = pred["instances"].to("cpu")

        segments = []
        all_pixels = []
        if len(instances) > 0:
            masks = instances.pred_masks.numpy()
            scores = instances.scores.numpy()
            for idx, (seg_mask, score) in enumerate(zip(masks, scores)):
                rows, cols = np.where(seg_mask)
                pixel_coords = [(int(x1 + c), int(y1 + r)) for r, c in zip(rows, cols)]
                all_pixels.extend(pixel_coords)
                segments.append({
                    "mask_id": idx,
                    "score": float(score),
                    "num_pixels": len(pixel_coords),
                    "pixel_coordinates": pixel_coords,
                })

        outputs.append({
            "tooth_id": data["class_name"],
            "crop_coords": [int(x1), int(y1), int(x2), int(y2)],
            "num_segments": len(segments),
            "segments": segments,
            "pixel_coordinates": all_pixels,
            "total_pixels": len(all_pixels),
            "detectron_output": pred,
        })

    return outputs


def save_bbox_visualization(pano_img_rgb, list_of_masks, output_path):
    vis = pano_img_rgb.copy()
    for m in list_of_masks:
        x1, y1, x2, y2 = map(int, m["bbox"])
        label = str(m["class_name"])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    fig = plt.figure(figsize=(12, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Bounding Boxes with Tooth ID")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_mask_overlay(pano_img_rgb, list_seg_tooth, output_path, alpha=0.5):
    vis = pano_img_rgb.copy()

    for result in list_seg_tooth:
        x1, y1, x2, y2 = result["crop_coords"]
        pred = result["detectron_output"]
        instances = pred["instances"].to("cpu")

        if len(instances) == 0:
            continue

        masks = instances.pred_masks.numpy()
        for mask in masks:
            h, w = mask.shape
            color = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)

            color_mask = np.zeros((h, w, 3), dtype=np.uint8)
            color_mask[mask] = color

            crop = vis[y1:y2, x1:x2]
            blended = np.where(
                mask[:, :, None],
                cv2.addWeighted(crop, 1 - alpha, color_mask, alpha, 0),
                crop,
            )
            vis[y1:y2, x1:x2] = blended

    fig = plt.figure(figsize=(16, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Mask Overlay")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_results_json(case_num, list_of_masks, list_seg_tooth, output_dir):
    result = {
        "case_number": str(case_num),
        "num_teeth_detected": len(list_of_masks),
        "teeth_data": [],
    }

    for mask_data, seg_data in zip(list_of_masks, list_seg_tooth):
        result["teeth_data"].append({
            "tooth_id": mask_data["class_name"],
            "confidence": float(mask_data["confidence"]),
            "bbox": [float(v) for v in mask_data["bbox"]],
            "crop_coords": seg_data["crop_coords"],
            "num_segments": seg_data["num_segments"],
            "total_pixels": seg_data["total_pixels"],
            "pixel_coordinates": seg_data["pixel_coordinates"],
            "segments_detail": seg_data["segments"],
        })

    with open(output_dir / f"case_{case_num}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def process_case(case_dir, yolo_model, detectron_predictor, output_root):
    case_num = case_dir.name.replace("case ", "")
    png_path = case_dir / f"case_{case_num}.png"
    if not png_path.exists():
        png_files = sorted(case_dir.glob("*.png"))
        if not png_files:
            return False, "No PNG found"
        png_path = png_files[0]

    img_bgr = cv2.imread(str(png_path))
    if img_bgr is None:
        return False, "Image read failed"

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    list_of_masks = process_panoramic_image(img_rgb, yolo_model)
    if len(list_of_masks) == 0:
        return False, "No teeth detected"

    list_seg_tooth = crop_and_segment_teeth(img_rgb, list_of_masks, detectron_predictor)

    case_out = output_root / f"case {case_num}"
    case_out.mkdir(parents=True, exist_ok=True)

    save_bbox_visualization(img_rgb, list_of_masks, case_out / f"case_{case_num}_bounding_boxes.png")
    save_mask_overlay(img_rgb, list_seg_tooth, case_out / f"case_{case_num}_mask_overlay.png")
    save_results_json(case_num, list_of_masks, list_seg_tooth, case_out)

    return True, f"OK ({len(list_of_masks)} teeth)"


def run_inference_on_raw_dataset(max_cases=None):
    project_root = Path.cwd()
    # Adjusting paths relative to the current working directory
    raw_data_dir = project_root.parent / "data"
    cases_root = raw_data_dir / "500 cases with annotation"

    if not cases_root.exists():
        raise FileNotFoundError(f"Cases folder not found: {cases_root}")

    output_root = project_root / "segmentation+recognition-dataset"
    output_root.mkdir(parents=True, exist_ok=True)

    yolo_model, detectron_predictor = initialize_models(raw_data_dir)

    case_dirs = sorted(
        [d for d in cases_root.iterdir() if d.is_dir() and d.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    if max_cases is not None:
        case_dirs = case_dirs[:max_cases]
        print(f"Limiting execution to {max_cases} sample cases.")

    ok_count = 0
    fail_count = 0
    print(f"Processing cases: {len(case_dirs)}")

    for case_dir in tqdm(case_dirs, desc="raw_data cases", unit="case"):
        try:
            ok, msg = process_case(case_dir, yolo_model, detectron_predictor, output_root)
            if ok:
                ok_count += 1
            else:
                fail_count += 1
        except Exception as e:
            fail_count += 1
            print(f"Error in {case_dir.name}: {e}")

    summary = {
        "source": "data/500 cases with annotation-raw",
        "processed": ok_count,
        "failed": fail_count,
        "total": len(case_dirs),
    }

    with open(output_root / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nDone")
    print(f"Output folder: {output_root.resolve()}")
    print(json.dumps(summary, indent=2))


# RUN ON ALL CASES
run_inference_on_raw_dataset()

# Uncomment the line below to run on ALL cases
# run_inference_on_raw_dataset()

d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cuda | GPU: NVIDIA GeForce RTX 4080 Laptop GPU
YOLO weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_pano_20250319.pt
Detectron2 weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_crop_20250424.pth


d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\fvcore\common\checkpoint.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=t

Models initialized successfully
Processing cases: 500


raw_data cases:   0%|          | 0/500 [00:00<?, ?case/s]d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
raw_data cases:   3%|▎         | 13/500 [01:41<1:03:18,  7.80s/case]


KeyboardInterrupt: 

## 2. Map tooth positions to caries regions (JSON output)
Combined script that processes dental cases to:
1. Map tooth positions to caries regions (JSON output)
2. Generate visual alignment debug images (PNG output)

Data Sources:
- JSON: segmentation+recognition-dataset/case X/case_X_results.json
  Contains pixel_coordinates (list of [x, y]) for each tooth
- ROI Image: raw_data/500-roi/case_X.png
  Binary mask where non-zero = caries, 0 = normal

Output (single root folder):
- caries_mapping_output/case X/case_X_caries_mapping.json
- caries_mapping_output/case X/case_X_alignment_detailed.png
- caries_mapping_output/caries_mapping_results.csv (summary)

In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


# =============================================================================
# ROI & JSON Loading Functions
# =============================================================================

def load_roi_image(roi_path: Path):
    """Load ROI mask and normalize to 2D grayscale array."""
    roi_img = cv2.imread(str(roi_path), cv2.IMREAD_UNCHANGED)
    if roi_img is None:
        raise FileNotFoundError(f"Could not load ROI image: {roi_path}")

    # Some PNGs are loaded as (H, W, 1); convert to strict 2D for downstream ops.
    if roi_img.ndim == 3:
        if roi_img.shape[2] == 1:
            roi_img = roi_img[:, :, 0]
        else:
            roi_img = cv2.cvtColor(roi_img, cv2.COLOR_BGR2GRAY)

    return roi_img


def load_tooth_json(json_path: Path):
    """Load tooth segmentation JSON with pixel coordinates."""
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# Caries Mapping Functions
# =============================================================================

def calculate_caries_overlap(roi_img, pixel_coordinates):
    """Calculate overlap between tooth pixels and caries region in ROI."""
    if not pixel_coordinates:
        return 0, 0, 0.0, []

    height, width = roi_img.shape[:2]
    caries_count = 0
    caries_coords = []
    valid_pixels = 0

    for coord in pixel_coordinates:
        x, y = coord[0], coord[1]

        if 0 <= x < width and 0 <= y < height:
            valid_pixels += 1
            if roi_img[y, x] > 0:
                caries_count += 1
                caries_coords.append([x, y])

    percentage = (caries_count / valid_pixels * 100) if valid_pixels > 0 else 0.0
    return caries_count, valid_pixels, percentage, caries_coords


def generate_caries_mapping(roi_img, tooth_data, case_num):
    """Generate per-tooth caries mapping results for one case."""
    results = []

    for tooth in tooth_data.get("teeth_data", []):
        tooth_id = tooth.get("tooth_id", "unknown")
        pixel_coords = tooth.get("pixel_coordinates", [])
        confidence = tooth.get("confidence", 0.0)

        caries_count, total_pixels, percentage, caries_coords = calculate_caries_overlap(
            roi_img, pixel_coords
        )

        results.append(
            {
                "case_number": int(case_num),
                "tooth_id": tooth_id,
                "confidence": float(confidence),
                "total_pixels": int(total_pixels),
                "caries_pixels": int(caries_count),
                "caries_percentage": round(float(percentage), 4),
                "has_caries": bool(caries_count > 0),
                "caries_coordinates": caries_coords,
            }
        )

    return results


# =============================================================================
# Visual Alignment Functions
# =============================================================================

def create_alignment_visualization(roi_img, tooth_data, alpha=0.5):
    """Create blended visualization: ROI background + tooth pixels overlay."""
    height, width = roi_img.shape[:2]

    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)
    roi_bgr = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    tooth_overlay = np.zeros((height, width, 3), dtype=np.uint8)

    colors = [
        (0, 255, 0),
        (0, 255, 255),
        (255, 255, 0),
        (0, 165, 255),
        (255, 0, 255),
        (128, 255, 0),
        (255, 128, 0),
        (0, 128, 255),
    ]

    tooth_count = 0
    total_pixels_drawn = 0
    out_of_bounds = 0

    for tooth in tooth_data.get("teeth_data", []):
        pixel_coords = tooth.get("pixel_coordinates", [])
        color = colors[tooth_count % len(colors)]
        tooth_count += 1

        for coord in pixel_coords:
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_overlay[y, x] = color
                total_pixels_drawn += 1
            else:
                out_of_bounds += 1

    tooth_mask = np.any(tooth_overlay > 0, axis=2)
    result = roi_bgr.copy()
    blended = cv2.addWeighted(roi_bgr, 1 - alpha, tooth_overlay, alpha, 0)
    result[tooth_mask] = blended[tooth_mask]
    strong_overlay = cv2.addWeighted(result, 0.7, tooth_overlay, 0.3, 0)
    result[tooth_mask] = strong_overlay[tooth_mask]

    return result, {
        "teeth_count": tooth_count,
        "pixels_drawn": total_pixels_drawn,
        "out_of_bounds": out_of_bounds,
        "roi_shape": (height, width),
    }


def create_detailed_alignment_image(roi_img, tooth_data):
    """Create 3-panel alignment image for QA/debug."""
    height, width = roi_img.shape[:2]
    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)

    roi_colored = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    caries_mask = roi_img > 0
    roi_colored[caries_mask] = [0, 0, 255]

    tooth_only = np.zeros((height, width, 3), dtype=np.uint8)
    for tooth in tooth_data.get("teeth_data", []):
        for coord in tooth.get("pixel_coordinates", []):
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_only[y, x] = [0, 255, 0]

    blended, stats = create_alignment_visualization(roi_img, tooth_data, alpha=0.6)

    tooth_mask = np.any(tooth_only > 0, axis=2)
    overlap_pixels = int(np.sum(tooth_mask & caries_mask))
    caries_pixels = int(np.sum(caries_mask))
    tooth_pixels = int(np.sum(tooth_mask))

    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(roi_colored, "ROI (Red=Caries)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(tooth_only, "Teeth (Green)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(blended, "ALIGNMENT CHECK", (20, 60), font, 1.5, (0, 255, 255), 3)

    stats_text = [
        f"Teeth: {stats['teeth_count']}",
        f"Tooth pixels: {tooth_pixels:,}",
        f"Caries pixels: {caries_pixels:,}",
        f"Overlap: {overlap_pixels:,}",
    ]
    y_offset = 120
    for text in stats_text:
        cv2.putText(blended, text, (20, y_offset), font, 1.0, (255, 255, 0), 2)
        y_offset += 40

    scale = 0.5
    roi_small = cv2.resize(roi_colored, None, fx=scale, fy=scale)
    tooth_small = cv2.resize(tooth_only, None, fx=scale, fy=scale)
    blended_small = cv2.resize(blended, None, fx=scale, fy=scale)
    combined = np.hstack([roi_small, tooth_small, blended_small])

    overlap_stats = {
        "overlap_pixels": overlap_pixels,
        "caries_pixels": caries_pixels,
        "tooth_pixels": tooth_pixels,
    }
    return combined, stats, overlap_stats


# =============================================================================
# Pipeline Functions for Current Project Structure
# =============================================================================

def parse_case_num_from_json(json_path: Path):
    """Extract numeric case id from filename: case_123_results.json -> 123."""
    stem = json_path.stem
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected JSON filename format: {json_path.name}")
    return int(parts[1])


def collect_case_records(segmentation_root: Path, roi_root: Path):
    """Collect case records from Step 1 output and matching ROI paths."""
    records = []

    case_dirs = sorted(
        [p for p in segmentation_root.iterdir() if p.is_dir() and p.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    for case_dir in case_dirs:
        json_candidates = sorted(case_dir.glob("case_*_results.json"))
        if not json_candidates:
            continue

        json_path = json_candidates[0]
        case_num = parse_case_num_from_json(json_path)
        roi_path = roi_root / f"case_{case_num}.png"

        records.append(
            {
                "case_number": case_num,
                "json_path": json_path,
                "roi_path": roi_path,
            }
        )

    return records


def process_single_case(record, output_root: Path):
    """Process one case and save both JSON + alignment image in one case folder."""
    case_num = record["case_number"]
    json_path = record["json_path"]
    roi_path = record["roi_path"]

    if not json_path.exists():
        return False, "JSON not found", None
    if not roi_path.exists():
        return False, f"ROI not found: {roi_path.name}", None

    roi_img = load_roi_image(roi_path)
    tooth_data = load_tooth_json(json_path)
    caries_results = generate_caries_mapping(roi_img, tooth_data, case_num)

    case_output_dir = output_root / f"case {case_num}"
    case_output_dir.mkdir(parents=True, exist_ok=True)

    caries_json_path = case_output_dir / f"case_{case_num}_caries_mapping.json"
    with open(caries_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "case_number": int(case_num),
                "teeth_caries_data": caries_results,
            },
            f,
            indent=2,
        )

    combined_img, stats, overlap_stats = create_detailed_alignment_image(roi_img, tooth_data)
    alignment_img_path = case_output_dir / f"case_{case_num}_alignment_detailed.png"
    cv2.imwrite(str(alignment_img_path), combined_img)

    teeth_with_caries = sum(1 for r in caries_results if r["has_caries"])
    msg = f"{len(caries_results)} teeth, {teeth_with_caries} with caries"

    case_stats = {
        "stats": stats,
        "overlap_stats": overlap_stats,
        "caries_results": caries_results,
    }
    return True, msg, case_stats


def run_caries_mapping_pipeline(
    segmentation_root: Path,
    roi_root: Path,
    output_root: Path,
    max_cases=None,
):
    """Run mapping pipeline and keep all outputs under one root folder."""
    output_root.mkdir(parents=True, exist_ok=True)

    records = collect_case_records(segmentation_root, roi_root)
    if not records:
        raise FileNotFoundError(
            "No case JSON files found. Check segmentation_root path and Step 1 outputs."
        )

    if max_cases is not None:
        records = records[:max_cases]
        print(f"Limiting execution to {max_cases} sample cases.")

    summary_stats = {
        "total_cases": len(records),
        "processed_cases": 0,
        "failed_cases": 0,
        "total_teeth": 0,
        "teeth_with_caries": 0,
    }
    all_caries_results = []

    print("=" * 70)
    print("Dental Caries Mapping Pipeline")
    print("=" * 70)
    print(f"Segmentation JSON source: {segmentation_root}")
    print(f"ROI source: {roi_root}")
    print(f"Output root: {output_root}")
    print("=" * 70)

    pbar = tqdm(records, desc="Processing cases", unit="case")
    for record in pbar:
        case_num = record["case_number"]
        try:
            success, msg, payload = process_single_case(record, output_root=output_root)

            if success and payload:
                case_results = payload["caries_results"]
                summary_stats["processed_cases"] += 1
                summary_stats["total_teeth"] += len(case_results)
                summary_stats["teeth_with_caries"] += sum(
                    1 for r in case_results if r["has_caries"]
                )

                for r in case_results:
                    csv_result = {k: v for k, v in r.items() if k != "caries_coordinates"}
                    all_caries_results.append(csv_result)

                pbar.set_postfix_str(f"case {case_num}: {msg}")
            else:
                summary_stats["failed_cases"] += 1
                pbar.set_postfix_str(f"case {case_num}: FAIL - {msg}")

        except Exception as e:
            summary_stats["failed_cases"] += 1
            pbar.set_postfix_str(f"case {case_num}: ERROR - {str(e)[:40]}")

    if all_caries_results:
        df = pd.DataFrame(all_caries_results)
        csv_path = output_root / "caries_mapping_results.csv"
        df.to_csv(csv_path, index=False)
        print(f"\nResults CSV saved to: {csv_path}")

        summary_df = (
            df.groupby(["case_number"]).agg(
                {
                    "tooth_id": "count",
                    "caries_pixels": "sum",
                    "has_caries": "sum",
                    "caries_percentage": "mean",
                }
            ).rename(
                columns={
                    "tooth_id": "total_teeth",
                    "has_caries": "teeth_with_caries",
                    "caries_percentage": "avg_caries_percentage",
                }
            )
        )
        summary_csv_path = output_root / "caries_summary_by_case.csv"
        summary_df.to_csv(summary_csv_path)
        print(f"Summary CSV saved to: {summary_csv_path}")

    print("\n" + "=" * 70)
    print("Processing Complete")
    print("=" * 70)
    print(f"Total cases found: {summary_stats['total_cases']}")
    print(f"Processed: {summary_stats['processed_cases']}")
    print(f"Failed: {summary_stats['failed_cases']}")
    print(f"Total teeth analyzed: {summary_stats['total_teeth']}")
    print(f"Teeth with caries: {summary_stats['teeth_with_caries']}")
    if summary_stats["total_teeth"] > 0:
        pct = summary_stats["teeth_with_caries"] / summary_stats["total_teeth"] * 100
        print(f"Caries prevalence: {pct:.2f}%")
    print("=" * 70)

    return all_caries_results, summary_stats


# =============================================================================
# Notebook Runner (Current Project Environment)
# =============================================================================

project_root = Path.cwd()

# Step 1 output folder (refactored pipeline)
segmentation_root = project_root / "segmentation+recognition-dataset"
if not segmentation_root.exists():
    raise FileNotFoundError(
        f"Segmentation output folder not found: {segmentation_root}. Run Step 1 first."
    )

# ROI folder from raw_data
roi_root = project_root.parent / "data" / "500-roi"
if not roi_root.exists():
    raise FileNotFoundError(
        f"ROI folder not found: {roi_root}."
    )

# Single output root as requested
output_root = project_root / "caries_mapping_output"

# RUN ON ALL CASES
all_caries_results, summary_stats = run_caries_mapping_pipeline(
    segmentation_root=segmentation_root,
    roi_root=roi_root,
    output_root=output_root,
    # max_cases=5
)

# Uncomment the below block to run on ALL cases
# all_caries_results, summary_stats = run_caries_mapping_pipeline(
#     segmentation_root=segmentation_root,
#     roi_root=roi_root,
#     output_root=output_root,
# )

Limiting execution to 5 sample cases.
Dental Caries Mapping Pipeline
Segmentation JSON source: c:\Users\jaopi\Desktop\SP\phase2-1april\segmentation+recognition-dataset
ROI source: c:\Users\jaopi\Desktop\SP\data\500-roi
Output root: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output


Processing cases: 100%|██████████| 5/5 [00:08<00:00,  1.77s/case, case 5: 23 teeth, 7 with caries]


Results CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_mapping_results.csv
Summary CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_summary_by_case.csv

Processing Complete
Total cases found: 5
Processed: 5
Failed: 0
Total teeth analyzed: 145
Teeth with caries: 33
Caries prevalence: 22.76%


## 3. PCA-based Tooth Alignment + Surface Classification & Visualization

This module applies PCA-based alignment to each detected tooth and classifies
caries lesions into 3 anatomical surfaces using Diagonal-from-Centroid zone voting:
- **Occlusal** — pixels nearest to the crown (vertical zone)
- **Mesial**   — pixels nearest to the midline side (horizontal zone)
- **Distal**   — pixels nearest to the distal side (horizontal zone)

Supports multiple lesions per tooth (each disconnected cluster classified independently).

Visualization features:
- Aligned tooth polygon with 4-triangle zone overlay (diagonal from centroid)
- Caries pixels coloured by predicted surface
- Gold star (★) = tooth centroid; diagonal guide-lines to all 4 corners
- Per-instance PNG: `tooth_{id}_instance{n}_{surface}.png`

Output Structure:
```
PCA_Output/case_{X}/
├── case_{X}.json           ← per-tooth classification + all instances
└── tooth_{id}/
    ├── tooth_{id}_instance0_Distal.png
    └── tooth_{id}_instance1_Mesial.png
```


In [ ]:
# =========================================================
# PCA & Surface Classification [v4.5 + v4.6 in one run]
# =========================================================
# v4.5: X-thirds dominant zone
# v4.6: Diagonal-from-Centroid (4-triangle zone split)
#
# This cell defines all PCA alignment, noise removal, and
# surface classification functions used by the pipeline.
# It then runs Baseline and Run 3 classifiers on 500 cases.
# =========================================================

import os, json, math
import warnings
from pathlib import Path
from typing import Any, Callable

import pandas as pd
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit


# ----------------------------------------------------------
# Progress bar helper (no external dependencies)
# ----------------------------------------------------------
def _progress_bar(current, total, prefix='Progress', bar_length=30):
    """
    Print an inline text-based progress bar that overwrites itself.

    Args:
        current (int): Current step number (1-indexed).
        total (int): Total number of steps.
        prefix (str): Label displayed before the bar.
        bar_length (int): Character width of the bar.
    """
    fraction = current / max(total, 1)
    filled = int(bar_length * fraction)
    bar = chr(9608) * filled + chr(9617) * (bar_length - filled)
    print(
        f'\r   {prefix} [{bar}] {fraction*100:.0f}% ({current}/{total})',
        end='', flush=True,
    )
    if current >= total:
        print()  # newline when complete


# ==========================================
# 1. Path configuration
# ==========================================

# _SP_DIR: project root directory (one level above this notebook).
_SP_DIR    = Path.cwd().parent

# SEG_DIR / CARIES_DIR: input data directories.
SEG_DIR    = str(_SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition")
CARIES_DIR = str(_SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output")

# Classification thresholds.
MAX_TILT_DEG      = 45.0   # Clamp extreme PCA angles (bad tooth masks)
MIN_CLUSTER_SIZE  = 15     # Noise removal: drop clusters smaller than this
LEFT_BOUND  = 0.40   # X-split: Left zone  0.00-0.40
RIGHT_BOUND = 0.60   # X-split: Right zone 0.60-1.00 | Center = 0.40-0.60 (Occlusal)

# Colour palette for visualisation.
SURFACE_COLORS = {"Occlusal": "#E74C3C", "Mesial": "#3498DB",
                  "Distal": "#27AE60", "Other": "#2ECC71", -1: "#95A5A6"}


# ==========================================
# 2. Utility functions
# ==========================================

def is_upper_jaw(tooth_id):
    """
    Check whether a tooth belongs to the upper jaw (quadrant 1 or 2).

    Args:
        tooth_id (str or int): FDI tooth identifier.

    Returns:
        bool: True if upper jaw.
    """
    return int(str(tooth_id)[0]) in [1, 2]


def get_quadrant(tooth_id):
    """
    Extract the FDI quadrant (1-4) from a tooth identifier.

    Args:
        tooth_id (str or int): FDI tooth identifier.

    Returns:
        int: Quadrant number (1-4).
    """
    return int(str(tooth_id)[0])


def load_seg(case_id):
    """
    Load the tooth segmentation JSON for a single case.

    Args:
        case_id (int): Numeric case identifier (1-500).

    Returns:
        dict or None: Parsed segmentation data, or None if file is missing.
    """
    path = os.path.join(SEG_DIR, f"case {case_id}", f"case_{case_id}_results.json")
    if not os.path.exists(path):
        print(f"[ERROR] SEG: {path}")
        return None
    with open(path) as f:
        return json.load(f)


def load_caries(case_id):
    """
    Load the caries-to-tooth mapping JSON for a single case.

    Args:
        case_id (int): Numeric case identifier (1-500).

    Returns:
        dict or None: Parsed caries mapping data, or None if file is missing.
    """
    path = os.path.join(CARIES_DIR, f"case {case_id}", f"case_{case_id}_caries_mapping.json")
    if not os.path.exists(path):
        print(f"[ERROR] CARIES: {path}")
        return None
    with open(path) as f:
        return json.load(f)


def build_seg_map(seg_data):
    """
    Build a mapping from tooth ID to its pixel coordinates.

    Args:
        seg_data (dict): Parsed segmentation JSON containing 'teeth_data'.

    Returns:
        dict: Mapping of str(tooth_id) -> list of [x, y] coordinates.
    """
    return {str(t["tooth_id"]): t.get("pixel_coordinates", []) for t in seg_data.get("teeth_data", [])}


def get_caries_list(data):
    """
    Extract the caries data list from a parsed caries mapping JSON.

    Args:
        data (dict): Parsed caries mapping JSON.

    Returns:
        list[dict]: List of teeth with caries data.
    """
    return data.get("teeth_caries_data", [])


def compute_centroid(points):
    """
    Compute the centroid (mean x, mean y) of a 2D point set.

    Args:
        points (list or np.ndarray): Nx2 array of [x, y] coordinates.

    Returns:
        tuple: (centroid_x, centroid_y) as floats.
    """
    arr = np.array(points, dtype=np.float64)
    return float(np.mean(arr[:, 0])), float(np.mean(arr[:, 1]))


def get_bbox(pts):
    """
    Compute the axis-aligned bounding box of a 2D point set.

    Args:
        pts (list or np.ndarray): Nx2 array of [x, y] coordinates.

    Returns:
        tuple: (x_min, y_min, width, height).
    """
    p = np.array(pts, dtype=np.float64)
    bbox_min, bbox_max = np.min(p, 0), np.max(p, 0)
    return bbox_min[0], bbox_min[1], bbox_max[0] - bbox_min[0], bbox_max[1] - bbox_min[1]


# =============================================================================
# NOISE REMOVAL (from week7)
# Removes small connected components from caries point sets to filter noise.
# =============================================================================
def remove_small_clusters(caries_pts, min_cluster=MIN_CLUSTER_SIZE):
    """
    Remove noise from caries points by discarding small connected components.

    Creates a binary mask from the point set, finds connected components
    via OpenCV, and keeps only those with area >= min_cluster.

    Args:
        caries_pts (list or np.ndarray): Nx2 caries pixel coordinates.
        min_cluster (int): Minimum cluster size to keep.

    Returns:
        np.ndarray: Filtered caries points with small clusters removed.
    """
    if len(caries_pts) < min_cluster:
        return caries_pts
    pts = np.array(caries_pts, dtype=np.int32)
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    pad = 2
    w = x_max - x_min + 1 + 2 * pad
    h = y_max - y_min + 1 + 2 * pad
    # Create a binary mask and find connected components.
    mask = np.zeros((h, w), dtype=np.uint8)
    shifted = pts - np.array([x_min - pad, y_min - pad])
    mask[shifted[:, 1], shifted[:, 0]] = 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    # Keep only components larger than the minimum cluster size.
    keep = np.zeros_like(mask)
    for lbl in range(1, n_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_cluster:
            keep[labels == lbl] = 255
    ys, xs = np.where(keep > 0)
    if len(xs) == 0:
        return caries_pts
    # Convert back to original coordinate space.
    return np.column_stack([xs + x_min - pad, ys + y_min - pad]).astype(np.float64)


# =============================================================================
# PCA — 4-Rule Orientation (ported from week7 multi_zone_classifier.py)
# Aligns each tooth using PCA eigenvectors with 4 rules:
#   Rule 1: Select the axis with larger Y component as vertical
#   Rule 2: Enforce vertical direction based on jaw (upper/lower)
#   Rule 3: Enforce horizontal direction based on quadrant (L/R)
#   Rule 4: Clamp extreme rotations to prevent bad alignments
# =============================================================================
def perform_pca(points, tooth_id):
    """
    Perform PCA-based 4-rule tooth orientation alignment.

    Args:
        points (list or np.ndarray): Nx2 tooth pixel coordinates.
        tooth_id (str or int): FDI tooth identifier.

    Returns:
        tuple: (mean_center, rotation_angle, was_clamped)
            - mean_center (np.ndarray): Centroid of the point set.
            - rotation_angle (float): Rotation angle in radians.
            - was_clamped (bool): True if the angle was clamped to 0.
    """
    pts = np.array(points, dtype=np.float64).reshape(-1, 2)
    mean = np.mean(pts, axis=0)
    centered = pts - mean

    # Compute eigenvectors via PCA.
    _, eigvecs = cv2.PCACompute(centered.astype(np.float32), mean=None)
    primary_eigenvector = eigvecs[0].astype(np.float64)
    secondary_eigenvector = eigvecs[1].astype(np.float64)

    # Rule 1: Select the axis with the larger Y component as vertical.
    if abs(primary_eigenvector[1]) >= abs(secondary_eigenvector[1]):
        vertical_axis = primary_eigenvector.copy()
        horizontal_axis = secondary_eigenvector.copy()
    else:
        vertical_axis = secondary_eigenvector.copy()
        horizontal_axis = primary_eigenvector.copy()

    # Rule 2: Enforce vertical axis direction based on jaw position.
    upper = is_upper_jaw(tooth_id)
    if upper:
        if vertical_axis[1] < 0:
            vertical_axis = -vertical_axis
    else:
        if vertical_axis[1] > 0:
            vertical_axis = -vertical_axis

    # Rule 3: Enforce horizontal axis direction based on quadrant.
    quadrant = get_quadrant(tooth_id)
    if quadrant in [1, 4]:
        if horizontal_axis[0] < 0:
            horizontal_axis = -horizontal_axis
    else:
        if horizontal_axis[0] > 0:
            horizontal_axis = -horizontal_axis

    # Compute rotation angle to align vertical axis to 90 or -90 degrees.
    angle_from_x = math.atan2(vertical_axis[1], vertical_axis[0])
    if upper:
        target_angle = math.pi / 2
    else:
        target_angle = -math.pi / 2

    rotation_angle = target_angle - angle_from_x

    # Normalise rotation angle to [-pi, pi].
    while rotation_angle > math.pi:
        rotation_angle -= 2 * math.pi
    while rotation_angle < -math.pi:
        rotation_angle += 2 * math.pi

    # Rule 4: Clamp extreme rotations to prevent bad alignments.
    clamped = False
    if abs(math.degrees(rotation_angle)) > MAX_TILT_DEG:
        rotation_angle = 0.0
        clamped = True

    return mean, rotation_angle, clamped


def rotate(pts, center, angle):
    """
    Rotate 2D points around a centre point by a given angle.

    Args:
        pts (list or np.ndarray): Nx2 array of [x, y] coordinates.
        center (np.ndarray): 1x2 centre of rotation.
        angle (float): Rotation angle in radians.

    Returns:
        np.ndarray: Rotated Nx2 array of coordinates.
    """
    p = np.array(pts, dtype=np.float64) - center
    c, s = np.cos(angle), np.sin(angle)
    return np.dot(p, np.array([[c, -s], [s, c]]).T) + center


# Type alias for classifier function signatures.
ClassifierFn = Callable[[str, list, list], tuple[str, float, dict[str, Any]]]


# =============================================================================
# CLASSIFICATION v4.5 — X-thirds dominant zone (week7 approach)
# Splits the tooth horizontally into three zones (Left/Center/Right)
# and assigns the surface based on which zone has the most caries pixels.
# =============================================================================
def classify_surface_v45(tooth_id: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    Classify caries surface using X-thirds dominant zone method.

    FDI anatomy reference (after 4-rule PCA, horizontal axis enforced):
      Q1/Q4 (image left):  Left third = Distal | Center = Occlusal | Right third = Mesial
      Q2/Q3 (image right): Left third = Mesial | Center = Occlusal | Right third = Distal

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle_deg, vote_fractions).
    """
    # Step 1: Noise removal.
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    # Step 2: PCA alignment (4-rule).
    center, angle, clamped = perform_pca(tooth_pts, tooth_id)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    rel_xs = np.clip((caries_rot[:, 0] - x) / w, 0.0, 1.0)
    n_pts  = len(rel_xs)

    # Step 3: X-thirds zone assignment.
    # After Rule 3 enforcement, horizontal axis is consistent per quadrant:
    #   Q1/Q4: +X is toward Mesial -> right third = Mesial, left third = Distal
    #   Q2/Q3: -X is toward Mesial -> left third = Mesial, right third = Distal
    quadrant = get_quadrant(tooth_id)
    if quadrant in [1, 4]:
        d_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        m_mask = rel_xs > RIGHT_BOUND
    else:
        m_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        d_mask = rel_xs > RIGHT_BOUND

    m_count = int(np.sum(m_mask))  # Mesial
    c_count = int(np.sum(c_mask))  # Occlusal (center)
    d_count = int(np.sum(d_mask))  # Distal

    # Step 4: Dominant zone wins.
    vote_map = {"Mesial": m_count, "Occlusal": c_count, "Distal": d_count}
    winner   = max(vote_map, key=vote_map.get)

    vote_fractions = {k: round(v / max(n_pts, 1), 4) for k, v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    return winner, float(math.degrees(angle)), vote_fractions


# =============================================================================
# CLASSIFICATION v4.6 — Diagonal-from-Centroid (4-triangle zone split)
# Draws diagonals through the tooth bbox centroid, creating 4 triangles.
# Horizontal triangles -> Mesial/Distal by quadrant, vertical -> Occlusal.
# =============================================================================
def classify_surface_v46(tooth_id: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    Classify caries surface using diagonal-from-centroid method.

    After 4-rule PCA rotation, splits by diagonals crossing the bbox centroid.
    Horizontal triangles -> Mesial/Distal by quadrant.
    Vertical triangles -> Occlusal.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle_deg, vote_fractions).
    """
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tooth_id)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    # Compute the bbox centroid for diagonal splitting.
    cx = x + (w / 2.0)
    cy = y + (h / 2.0)
    quadrant = get_quadrant(tooth_id)

    vote_map = {"Mesial": 0, "Occlusal": 0, "Distal": 0}

    for px, py in caries_rot:
        dx = float(px - cx)
        dy = float(py - cy)

        # Determine zone: left/right triangles vs top/bottom triangles.
        if abs(dx / w) > abs(dy / h):
            if quadrant in [1, 4]:
                zone = "Mesial" if dx > 0 else "Distal"
            else:
                zone = "Distal" if dx > 0 else "Mesial"
        else:
            zone = "Occlusal"

        vote_map[zone] += 1

    winner = max(vote_map, key=vote_map.get)
    n_pts = len(caries_rot)

    vote_fractions = {k: round(v / max(n_pts, 1), 4) for k, v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    return winner, float(math.degrees(angle)), vote_fractions


# =============================================================================
# CLASSIFICATION classify_biaxial — adds Y-axis crown zone (top 30% -> Occlusal)
# Enhances X-thirds by boosting the Occlusal vote when the majority of
# caries pixels fall within the crown zone (top/bottom 30% depending on jaw).
# =============================================================================
def classify_biaxial(tooth_id: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    Classify caries surface using biaxial (X-thirds + Y crown zone) method.

    Adds a Y-axis crown zone rule: if >50% of caries pixels are in the
    crown zone (top 30% for lower jaw / bottom 30% for upper jaw), the
    Occlusal vote is boosted.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle_deg, vote_fractions).
    """
    # Step 1: Noise removal and PCA alignment.
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tooth_id)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    # Step 2: Compute normalised X and Y positions.
    rel_xs = np.clip((caries_rot[:, 0] - x) / w, 0.0, 1.0)
    rel_ys = np.clip((caries_rot[:, 1] - y) / h, 0.0, 1.0)
    n_pts  = len(rel_xs)

    quadrant = get_quadrant(tooth_id)

    # Step 3: X-thirds zone assignment (same as v4.5).
    if quadrant in [1, 4]:
        d_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        m_mask = rel_xs > RIGHT_BOUND
    else:
        m_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        d_mask = rel_xs > RIGHT_BOUND

    m_count = int(np.sum(m_mask))
    c_count = int(np.sum(c_mask))
    d_count = int(np.sum(d_mask))

    # Step 4: Y-axis crown zone logic.
    # Upper jaw: crown is at the bottom of the image (Y >= 0.70).
    # Lower jaw: crown is at the top of the image (Y <= 0.30).
    is_upper = quadrant in [1, 2]
    if is_upper:
        crown_mask = rel_ys >= 0.70
    else:
        crown_mask = rel_ys <= 0.30

    crown_count = int(np.sum(crown_mask))

    # Step 5: Biaxial override — if >50% of pixels are in crown zone, boost Occlusal.
    if crown_count > (n_pts * 0.5):
        c_count += crown_count

    vote_map = {"Mesial": m_count, "Occlusal": c_count, "Distal": d_count}
    winner   = max(vote_map, key=vote_map.get)

    vote_fractions = {k: round(v / max(n_pts, 1), 4) for k, v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    vote_fractions["crown_ratio"] = round(crown_count / max(n_pts, 1), 4)
    return winner, float(math.degrees(angle)), vote_fractions


# =============================================================================
# CLASSIFICATION classify_fuzzy — Euclidean distance-based fuzzy voting
# Weights each pixel's vote inversely proportional to its squared distance
# from three focal points (center=Occlusal, left/right=Mesial/Distal).
# =============================================================================
def classify_fuzzy(tooth_id: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    Classify caries surface using fuzzy distance-weighted voting.

    Each pixel votes for all three zones simultaneously, weighted by
    inverse squared Euclidean distance to focal points.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle_deg, vote_fractions).
    """
    # Step 1: Noise removal and PCA alignment.
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tooth_id)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    # Step 2: Compute normalised X and Y positions.
    rel_xs = np.clip((caries_rot[:, 0] - x) / w, 0.0, 1.0)
    rel_ys = np.clip((caries_rot[:, 1] - y) / h, 0.0, 1.0)
    n_pts  = len(rel_xs)

    quadrant = get_quadrant(tooth_id)
    is_upper = quadrant in [1, 2]

    # Step 3: Define focal points.
    # Upper jaw: crown at bottom (Y=0.85); lower jaw: crown at top (Y=0.15).
    crown_y = 0.85 if is_upper else 0.15
    focal_center = (0.5, crown_y)   # Occlusal focal point
    focal_left   = (0.0, 0.5)       # Left-edge focal point
    focal_right  = (1.0, 0.5)       # Right-edge focal point

    vote_left = 0.0
    vote_center = 0.0
    vote_right = 0.0

    # Step 4: Fuzzy voting — inverse squared distance weighting.
    for rx, ry in zip(rel_xs, rel_ys):
        # Squared Euclidean distance to each focal point.
        d2_center = (rx - focal_center[0])**2 + (ry - focal_center[1])**2
        d2_left   = (rx - focal_left[0])**2   + (ry - focal_left[1])**2
        d2_right  = (rx - focal_right[0])**2  + (ry - focal_right[1])**2

        # Weight inversely proportional to distance (epsilon=0.05 prevents div-by-zero).
        vote_center += 1.0 / (d2_center + 0.05)
        vote_left   += 1.0 / (d2_left + 0.05)
        vote_right  += 1.0 / (d2_right + 0.05)

    # Step 5: Map left/right votes to Mesial/Distal based on quadrant.
    if quadrant in [1, 4]:
        vote_map = {"Distal": vote_left, "Occlusal": vote_center, "Mesial": vote_right}
    else:
        vote_map = {"Mesial": vote_left, "Occlusal": vote_center, "Distal": vote_right}

    winner = max(vote_map, key=vote_map.get)

    # Convert raw scores to percentages for logging.
    total_votes = sum(vote_map.values())
    vote_fractions = {k: round(v / total_votes, 4) for k, v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped

    return winner, float(math.degrees(angle)), vote_fractions


# =============================================================================
# Run 3 (classify_ml) — initial RF-based classifier defined in cell 5.
# This version uses 14 features (including y_max).  The cleaned version
# in cell 7 uses 13 features.  Both share the same rf_model global.
# =============================================================================

# FEATURE_COLS for this cell's version (14 features including y_max).
FEATURE_COLS = [
    'is_upper', 'x_mean', 'y_mean', 'x_std', 'y_std',
    'x_min', 'x_max', 'y_min', 'y_max', 'x_range', 'y_range',
    'x_centroid_dist', 'aspect_ratio', 'coverage',
]

# rf_model: the trained Random Forest classifier (shared global).
rf_model = None

# RUN3_MODEL_PATH: path to the serialised model pickle.
RUN3_MODEL_PATH = Path.cwd() / 'rf_classify_ml.pkl'

# RUN3_OUTPUT_ROOT: directory for Run 3 prediction outputs.
RUN3_OUTPUT_ROOT = Path.cwd() / 'PCA_Output_Run3'
RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts):
    """
    Extract a dictionary of geometric features for one caries-tooth pair.

    This version extracts 14 features (including y_max).  The cleaned
    Run 3 cell uses a 13-feature version without y_max.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        dict or None: Feature dictionary, or None if invalid input.
    """
    # Remove small noise clusters from caries points.
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return None

    # PCA-align tooth and caries points.
    center, angle, _ = perform_pca(tooth_pts, tooth_id)
    tooth_rot = rotate(tooth_pts, center, angle)
    caries_rot = rotate(caries_clean, center, angle)

    # Compute bounding box extents.
    xy_min = np.min(tooth_rot, axis=0)
    xy_range = np.ptp(tooth_rot, axis=0)
    if xy_range[0] <= 0 or xy_range[1] <= 0:
        return None

    # Normalise caries coordinates to [0, 1] relative to tooth bbox.
    x_rel = np.clip((caries_rot[:, 0] - xy_min[0]) / xy_range[0], 0.0, 1.0)
    y_rel = np.clip((caries_rot[:, 1] - xy_min[1]) / xy_range[1], 0.0, 1.0)

    return {
        'is_upper': 1 if int(str(tooth_id)[0]) in [1, 2] else 0,
        'x_mean': float(np.mean(x_rel)),
        'y_mean': float(np.mean(y_rel)),
        'x_std': float(np.std(x_rel)),
        'y_std': float(np.std(y_rel)),
        'x_min': float(np.min(x_rel)),
        'x_max': float(np.max(x_rel)),
        'y_min': float(np.min(y_rel)),
        'y_max': float(np.max(y_rel)),
        'x_range': float(np.max(x_rel) - np.min(x_rel)),
        'y_range': float(np.max(y_rel) - np.min(y_rel)),
        'x_centroid_dist': float(abs(np.mean(x_rel) - 0.5)),
        'aspect_ratio': float(xy_range[0] / xy_range[1]),
        'coverage': float(len(caries_clean) / (len(tooth_pts) + 1e-6)),
    }


def _ensure_rf_model():
    """
    Ensure the Random Forest model is loaded into the rf_model global.

    Loads from disk if not already in memory.

    Returns:
        RandomForestClassifier or None: The loaded model, or None if missing.
    """
    global rf_model
    if rf_model is not None:
        return rf_model
    if RUN3_MODEL_PATH.exists():
        rf_model = joblib.load(RUN3_MODEL_PATH)
    return rf_model


def classify_ml(tooth_id, tooth_pts, caries_pts):
    """
    Classify caries surface using the Random Forest model with proba filtering.

    Falls back to classify_surface_v45 if the model or features are unavailable.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle, metadata_dict).
    """
    model = _ensure_rf_model()
    if model is None:
        return classify_surface_v45(tooth_id, tooth_pts, caries_pts)

    features = _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts)
    if features is None:
        return classify_surface_v45(tooth_id, tooth_pts, caries_pts)

    # Build a single-row DataFrame matching the training feature schema.
    prediction_input_df = pd.DataFrame([[features[col] for col in FEATURE_COLS]], columns=FEATURE_COLS)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            class_probabilities = model.predict_proba(prediction_input_df)[0]
        model_classes = list(model.classes_)
        # Filter to only valid anatomical surfaces.
        valid_surface_classes = ['Occlusal', 'Mesial', 'Distal']
        surface_scores = {cls: class_probabilities[model_classes.index(cls)] for cls in valid_surface_classes if cls in model_classes}
        if not surface_scores:
            return classify_surface_v45(tooth_id, tooth_pts, caries_pts)
        winner = max(surface_scores, key=surface_scores.get)
        return winner, 0.0, {'method': 'RandomForest_Proba'}
    except Exception:
        return classify_surface_v45(tooth_id, tooth_pts, caries_pts)


def process_case_ml(case_id: int, output_root: str | Path = RUN3_OUTPUT_ROOT):
    """
    Run the ML classifier on all caries-affected teeth in one case
    and save predictions as a JSON file.

    Args:
        case_id (int): Numeric case identifier (1-500).
        output_root (str or Path): Root directory for prediction JSONs.

    Returns:
        tuple: (is_success, status_message).
    """
    seg_data = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None:
        return False, 'missing input files'

    out_dir = Path(output_root) / f'case_{case_id}'
    out_dir.mkdir(parents=True, exist_ok=True)

    segmentation_map = build_seg_map(seg_data)
    results = {'case_number': case_id, 'teeth_data': []}
    for tooth in get_caries_list(caries_data):
        tooth_id = str(tooth.get('tooth_id'))
        tooth_pts = segmentation_map.get(tooth_id, [])
        caries_pts = tooth.get('caries_coordinates', [])
        surface, angle, vote_fractions = classify_ml(tooth_id, tooth_pts, caries_pts)
        results['teeth_data'].append({
            'tooth_id': tooth_id,
            'predicted_surface_fine': surface,
            'caries_position_detail': surface,
            'rotation_angle_deg': angle,
            'vote_fractions': vote_fractions,
            'tooth_coordinates': tooth_pts,
            'caries_coordinates': caries_pts,
        })

    with open(out_dir / f'case_{case_id}.json', 'w') as f:
        json.dump(results, f, indent=2)
    return True, f"saved {case_id}"


# =============================================================================
# End of classification methods
# =============================================================================


def compute_caries_stats(caries_pts, tooth_pts):
    """
    Compute basic caries statistics: pixel count and percentage coverage.

    Args:
        caries_pts (list): Caries pixel coordinates.
        tooth_pts (list): Tooth pixel coordinates.

    Returns:
        tuple: (caries_pixel_count, coverage_percentage).
    """
    cp, tp = len(caries_pts), len(tooth_pts)
    return cp, (cp / tp * 100) if tp else 0


def visualize_tooth_with_zones(
    tooth_pts,
    caries_pts,
    tooth_id,
    classification,
    version: str,
    vote_fractions: dict[str, Any] | None = None,
    save_path: str | Path | None = None,
) -> plt.Figure:
    """
    Create a visual plot of a tooth with classification zone overlays.

    Draws the tooth point cloud, caries overlay, zone boundaries,
    and classification result.  Saves to disk if save_path is provided.

    Args:
        tooth_pts (np.ndarray): Nx2 tooth coordinates (already rotated).
        caries_pts (np.ndarray): Nx2 caries coordinates (already rotated).
        tooth_id (str): FDI tooth identifier.
        classification (str): Predicted surface label.
        version (str): Version tag ('v4.5' or 'v4.6').
        vote_fractions (dict, optional): Vote fractions for title display.
        save_path (str or Path, optional): File path to save the figure.

    Returns:
        plt.Figure: The matplotlib figure object.
    """
    tooth_pts  = np.array(tooth_pts,  dtype=np.float64)
    caries_pts = np.array(caries_pts, dtype=np.float64)
    x, y = np.min(tooth_pts, axis=0)
    w, h = np.ptp(tooth_pts, axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))
    q = get_quadrant(tooth_id)

    # Draw zone boundaries based on the classification version.
    if version == "v4.5":
        left_col  = SURFACE_COLORS["Distal"]  if q in [1, 4] else SURFACE_COLORS["Mesial"]
        right_col = SURFACE_COLORS["Mesial"] if q in [1, 4] else SURFACE_COLORS["Distal"]
        t1 = x + w * LEFT_BOUND
        t2 = x + w * RIGHT_BOUND
        ax.fill([x, t1, t1, x],          [y, y, y + h, y + h], alpha=0.18, color=left_col)
        ax.fill([t1, t2, t2, t1],         [y, y, y + h, y + h], alpha=0.18, color=SURFACE_COLORS["Occlusal"])
        ax.fill([t2, x + w, x + w, t2],   [y, y, y + h, y + h], alpha=0.18, color=right_col)
    else:
        ax.plot([x, x + w], [y, y + h], linestyle="--", linewidth=1.2, color="gold", alpha=0.6)
        ax.plot([x, x + w], [y + h, y], linestyle="--", linewidth=1.2, color="gold", alpha=0.6)

    # Plot tooth and caries points.
    ax.scatter(tooth_pts[:, 0], tooth_pts[:, 1], c="gray", s=2, alpha=0.4)
    color = SURFACE_COLORS.get(classification, "#95A5A6")
    if len(caries_pts) > 0:
        ax.scatter(caries_pts[:, 0], caries_pts[:, 1], c=color, s=10, zorder=5)

    # Mark caries centroid.
    cx, cy = compute_centroid(caries_pts)
    ax.plot(cx, cy, "*", markersize=14, color="gold", zorder=6)
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ls="--", lw=1))

    title = f"Tooth {tooth_id} Q{q} \u2014 {classification} ({version})"
    if vote_fractions:
        v = vote_fractions
        title += f"\nOcc={v.get('Occlusal', 0):.2f} M={v.get('Mesial', 0):.2f} D={v.get('Distal', 0):.2f}"

    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    return fig


def process_case(case_id: int, out_dir: str, classifier: ClassifierFn, version: str) -> None:
    """
    Run a classifier on all caries-affected teeth in one case and save JSON.

    Args:
        case_id (int): Numeric case identifier (1-500).
        out_dir (str): Output directory for this version.
        classifier (ClassifierFn): The classification function to use.
        version (str): Version tag for labelling.
    """
    seg_data    = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None:
        return

    segmentation_map = build_seg_map(seg_data)
    teeth_caries_list = get_caries_list(caries_data)
    results = {"case_number": case_id, "teeth_data": []}

    for tooth in teeth_caries_list:
        tooth_id   = str(tooth["tooth_id"])
        caries_pts = tooth.get("caries_coordinates", [])
        tooth_pts  = segmentation_map.get(tooth_id, [])
        if len(caries_pts) == 0 or len(tooth_pts) < 10:
            continue

        surface, angle, vf = classifier(tooth_id, tooth_pts, caries_pts)
        pixels, pct = compute_caries_stats(caries_pts, tooth_pts)
        results["teeth_data"].append({
            "tooth_id": tooth_id,
            "version": version,
            "has_caries": True,
            "confidence": tooth.get("confidence", 0),
            "caries_position_detail": surface,
            "predicted_surface_fine": surface,
            "vote_fractions": vf,
            "rotation_angle": round(angle, 2),
            "tooth_coordinates": tooth_pts,
            "caries_coordinates": caries_pts,
            "caries_pixels": pixels,
            "caries_percentage": round(pct, 4)
        })

    case_dir = os.path.join(out_dir, f"case_{case_id}")
    os.makedirs(case_dir, exist_ok=True)
    with open(os.path.join(case_dir, f"case_{case_id}.json"), "w") as f:
        json.dump(results, f, indent=4)


def process_case_visual(case_id: int, out_dir: str, classifier: ClassifierFn, version: str) -> None:
    """
    Generate visual plots for all teeth in a processed case.

    Reads the prediction JSON from process_case(), rotates the coordinates
    via PCA, and saves a visualisation image per tooth.

    Args:
        case_id (int): Numeric case identifier (1-500).
        out_dir (str): Output directory for this version.
        classifier (ClassifierFn): Kept for pipeline signature symmetry.
        version (str): Version tag for labelling.
    """
    _ = classifier  # kept in signature by design for versioned pipeline symmetry

    case_dir = Path(out_dir) / f"case_{case_id}"
    jp = case_dir / f"case_{case_id}.json"
    if not jp.exists():
        return

    with open(jp) as f:
        data = json.load(f)

    for tooth in data["teeth_data"]:
        tooth_id = tooth["tooth_id"]
        tp = tooth["tooth_coordinates"]
        cp = tooth["caries_coordinates"]

        c, a, _ = perform_pca(tp, tooth_id)
        tr = rotate(tp, c, a)
        cr = rotate(np.array(cp, dtype=np.float64), c, a)

        td = case_dir / f"tooth_{tooth_id}"
        td.mkdir(parents=True, exist_ok=True)

        visualize_tooth_with_zones(
            tr,
            cr,
            tooth_id,
            tooth["caries_position_detail"],
            version=version,
            vote_fractions=tooth.get("vote_fractions", {}),
            save_path=td / f"tooth_{tooth_id}_visual.png",
        )


# ==========================================
# VERSION_CONFIGS — version registry
# ==========================================
VERSION_CONFIGS = {
    "Baseline": {
        "out_dir": "PCA_Output_Baseline",
        "classifier": classify_surface_v45,
        "label": "X-Thirds Hard Partition (Baseline)",
    },
    "Run3": {
        "out_dir": "PCA_Output_Run3",
        "classifier": classify_ml,
        "label": "Geometric Feature + ML Classifier",
    }
}


# ==========================================
# Execution loop — run all versions on 500 cases
# ==========================================
for version, cfg in VERSION_CONFIGS.items():
    print(f"\n{'=' * 60}")
    print(f"  Running {version} \u2014 {cfg['label']}")
    print(f"  Output \u2192 {cfg['out_dir']}/")
    print(f"{'=' * 60}")

    os.makedirs(cfg["out_dir"], exist_ok=True)

    for cid in range(1, 501):
        process_case(cid, cfg["out_dir"], cfg["classifier"], version)
        process_case_visual(cid, cfg["out_dir"], cfg["classifier"], version)
        # Update progress bar.
        _progress_bar(cid, 500, f"{version}")

print("\n[ALL DONE] Both versions complete.")

# RUN 3: Random Forest classify_ml (cleaned, SP-root paths)
ได้ rf_model ที่ขั้นตอนนี้

In [8]:
# =========================================================
# RUN 3: Random Forest classify_ml (cleaned, SP-root paths)
# =========================================================
# This cell builds, trains, and runs the Run 3 ML pipeline:
#   1. Define fallback helpers (PCA, bbox, clustering, etc.)
#   2. Extract 13 geometric features from 500 cases
#   3. Train a Random Forest classifier with GroupShuffleSplit
#   4. Predict caries surfaces for all 500 cases
#   5. Evaluate predictions against XML ground truth
# =========================================================

import warnings
import joblib
import json
import pandas as pd
import numpy as np
import math
import cv2
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
 )


# ----------------------------------------------------------
# Progress bar helper (no external dependencies)
# ----------------------------------------------------------
def _progress_bar(current, total, prefix='Progress', bar_length=30):
    """
    Print an inline text-based progress bar that overwrites itself.

    Args:
        current (int): Current step number (1-indexed).
        total (int): Total number of steps.
        prefix (str): Label displayed before the bar.
        bar_length (int): Character width of the bar.
    """
    fraction = current / max(total, 1)
    filled = int(bar_length * fraction)
    bar = chr(9608) * filled + chr(9617) * (bar_length - filled)
    print(
        f'\r   {prefix} [{bar}] {fraction*100:.0f}% ({current}/{total})',
        end='', flush=True,
    )
    if current >= total:
        print()  # newline when complete


# FEATURE_COLS: ordered list of the 13 geometric features used by the
# Random Forest model.  The order must match the training schema exactly.
FEATURE_COLS = [
    'is_upper', 'x_mean', 'y_mean', 'x_std', 'y_std',
    'x_min', 'x_max', 'y_min', 'x_range', 'y_range',
    'x_centroid_dist', 'aspect_ratio', 'coverage',
 ]

# rf_model: the loaded/trained sklearn RandomForestClassifier instance.
# Starts as None; populated by train_classify_ml() or loaded from pickle.
rf_model = None

# rf_feature_cols: reference to the feature column list used during training.
rf_feature_cols = FEATURE_COLS

# ==========================================
# 1. Path configuration (resolved from SP root)
# ==========================================

# _SP_DIR: project root directory (one level above this notebook).
_SP_DIR    = Path.cwd().parent

# SEG_DIR: path to the tooth segmentation + recognition output directory.
SEG_DIR    = str(_SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition")

# CARIES_DIR: path to the caries-to-tooth mapping output directory.
CARIES_DIR = str(_SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output")

# Keep Path objects used elsewhere for backwards compatibility.
RUN3_SEG_ROOT = Path(SEG_DIR)
RUN3_CARIES_ROOT = Path(CARIES_DIR)

# RUN3_GT_ROOT: path to XML ground-truth annotations (500 cases).
RUN3_GT_ROOT = _SP_DIR / "data" / "500 cases with annotation"

# RUN3_OUTPUT_ROOT: directory where per-case prediction JSONs are written.
RUN3_OUTPUT_ROOT = Path.cwd() / 'PCA_Output_Run3'
RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# VALID_SURFACES: the four surface classes used in evaluation.
VALID_SURFACES = ["Occlusal", "Mesial", "Distal", "Other"]

# ----------------------------------------------------------
# Fallback: parse_case_ground_truth
# Imports the XML parser from week6 when the evaluation cell
# has not been run yet.
# ----------------------------------------------------------
if "parse_case_ground_truth" not in globals():
    try:
        import sys
        _WEEK6_DIR = _SP_DIR / "week6"
        if str(_WEEK6_DIR) not in sys.path:
            sys.path.append(str(_WEEK6_DIR))
        import xml_ground_truth_parser as _gt_parser

        def parse_case_ground_truth(case_folder):
            """
            Parse all AIM XML ground-truth files in a case folder.

            Args:
                case_folder (Path or str): Directory containing *.xml files.

            Returns:
                list[dict]: Each dict has 'tooth' (str FDI) and 'surface' (str).
            """
            ground_truth_list = []
            for xml_file in sorted(Path(case_folder).glob("*.xml")):
                parsed = _gt_parser.parse_aim_xml(str(xml_file))
                if parsed is None:
                    continue
                tooth = str(parsed.get("tooth_fdi", parsed.get("tooth", "Unknown")))
                surface = parsed.get("surface_name", parsed.get("surface", "Other"))
                if surface not in VALID_SURFACES:
                    surface = "Other"
                ground_truth_list.append({"tooth": tooth, "surface": surface})
            return ground_truth_list
    except Exception as e:
        raise RuntimeError(
            "parse_case_ground_truth is missing and fallback import failed. "
            "Run the evaluation cell or check week6/xml_ground_truth_parser.py."
        ) from e

# ----------------------------------------------------------
# Fallback: is_upper_jaw
# ----------------------------------------------------------
if "is_upper_jaw" not in globals():
    def is_upper_jaw(tooth_id):
        """
        Check whether a tooth belongs to the upper jaw (quadrant 1 or 2).

        Args:
            tooth_id (str or int): FDI tooth identifier.

        Returns:
            bool: True if upper jaw.
        """
        return int(str(tooth_id)[0]) in [1, 2]

# ----------------------------------------------------------
# Fallback: get_quadrant
# ----------------------------------------------------------
if "get_quadrant" not in globals():
    def get_quadrant(tooth_id):
        """
        Extract the FDI quadrant (1-4) from a tooth identifier.

        Args:
            tooth_id (str or int): FDI tooth identifier.

        Returns:
            int: Quadrant number (1-4).
        """
        return int(str(tooth_id)[0])

# ----------------------------------------------------------
# Fallback: get_bbox
# ----------------------------------------------------------
if "get_bbox" not in globals():
    def get_bbox(pts):
        """
        Compute the axis-aligned bounding box of a 2D point set.

        Args:
            pts (list or np.ndarray): Nx2 array of [x, y] coordinates.

        Returns:
            tuple: (x_min, y_min, width, height).
        """
        p = np.array(pts, dtype=np.float64)
        bbox_min, bbox_max = np.min(p, axis=0), np.max(p, axis=0)
        return bbox_min[0], bbox_min[1], bbox_max[0] - bbox_min[0], bbox_max[1] - bbox_min[1]

# ----------------------------------------------------------
# Fallback: rotate
# ----------------------------------------------------------
if "rotate" not in globals():
    def rotate(pts, center, angle):
        """
        Rotate 2D points around a centre point by a given angle.

        Args:
            pts (list or np.ndarray): Nx2 array of [x, y] coordinates.
            center (np.ndarray): 1x2 centre of rotation.
            angle (float): Rotation angle in radians.

        Returns:
            np.ndarray: Rotated Nx2 array.
        """
        p = np.array(pts, dtype=np.float64) - center
        c, s = np.cos(angle), np.sin(angle)
        return np.dot(p, np.array([[c, -s], [s, c]]).T) + center

# ----------------------------------------------------------
# Fallback: remove_small_clusters
# ----------------------------------------------------------
if "remove_small_clusters" not in globals():
    MIN_CLUSTER_SIZE = 15
    def remove_small_clusters(caries_pts, min_cluster=MIN_CLUSTER_SIZE):
        """
        Remove noise from caries points by discarding small connected components.

        Args:
            caries_pts (list or np.ndarray): Nx2 caries pixel coordinates.
            min_cluster (int): Minimum cluster size to keep.

        Returns:
            np.ndarray: Filtered caries points.
        """
        if len(caries_pts) < min_cluster:
            return caries_pts
        pts = np.array(caries_pts, dtype=np.int32)
        x_min, y_min = pts.min(axis=0)
        x_max, y_max = pts.max(axis=0)
        pad = 2
        w = x_max - x_min + 1 + 2 * pad
        h = y_max - y_min + 1 + 2 * pad
        mask = np.zeros((h, w), dtype=np.uint8)
        shifted = pts - np.array([x_min - pad, y_min - pad])
        mask[shifted[:, 1], shifted[:, 0]] = 255
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        keep = np.zeros_like(mask)
        for lbl in range(1, n_labels):
            if stats[lbl, cv2.CC_STAT_AREA] >= min_cluster:
                keep[labels == lbl] = 255
        ys, xs = np.where(keep > 0)
        if len(xs) == 0:
            return caries_pts
        return np.column_stack([xs + x_min - pad, ys + y_min - pad]).astype(np.float64)

# ----------------------------------------------------------
# Fallback: perform_pca
# ----------------------------------------------------------
if "perform_pca" not in globals():
    MAX_TILT_DEG = 45.0
    def perform_pca(points, tooth_id):
        """
        PCA-based 4-rule tooth orientation alignment.

        Args:
            points (list or np.ndarray): Nx2 tooth pixel coordinates.
            tooth_id (str or int): FDI tooth identifier.

        Returns:
            tuple: (mean_center, rotation_angle, was_clamped).
        """
        pts = np.array(points, dtype=np.float64).reshape(-1, 2)
        mean = np.mean(pts, axis=0)
        centered = pts - mean

        _, eigvecs = cv2.PCACompute(centered.astype(np.float32), mean=None)
        primary_eigenvector = eigvecs[0].astype(np.float64)
        secondary_eigenvector = eigvecs[1].astype(np.float64)

        if abs(primary_eigenvector[1]) >= abs(secondary_eigenvector[1]):
            vertical_axis = primary_eigenvector.copy()
            horizontal_axis = secondary_eigenvector.copy()
        else:
            vertical_axis = secondary_eigenvector.copy()
            horizontal_axis = primary_eigenvector.copy()

        upper = is_upper_jaw(tooth_id)
        if upper:
            if vertical_axis[1] < 0:
                vertical_axis = -vertical_axis
        else:
            if vertical_axis[1] > 0:
                vertical_axis = -vertical_axis

        quadrant = get_quadrant(tooth_id)
        if quadrant in [1, 4]:
            if horizontal_axis[0] < 0:
                horizontal_axis = -horizontal_axis
        else:
            if horizontal_axis[0] > 0:
                horizontal_axis = -horizontal_axis

        angle_from_x = math.atan2(vertical_axis[1], vertical_axis[0])
        target_angle = math.pi / 2 if upper else -math.pi / 2
        rotation_angle = target_angle - angle_from_x

        while rotation_angle > math.pi:
            rotation_angle -= 2 * math.pi
        while rotation_angle < -math.pi:
            rotation_angle += 2 * math.pi

        clamped = False
        if abs(math.degrees(rotation_angle)) > MAX_TILT_DEG:
            rotation_angle = 0.0
            clamped = True

        return mean, rotation_angle, clamped

# ----------------------------------------------------------
# Fallback: build_seg_map
# ----------------------------------------------------------
if "build_seg_map" not in globals():
    def build_seg_map(seg_data):
        """
        Build a mapping from tooth ID to its pixel coordinates.

        Args:
            seg_data (dict): Parsed segmentation JSON containing 'teeth_data'.

        Returns:
            dict: Mapping of str(tooth_id) -> list of [x, y] coordinates.
        """
        return {
            str(t["tooth_id"]): t.get("pixel_coordinates", [])
            for t in seg_data.get("teeth_data", [])
        }

# ----------------------------------------------------------
# Fallback: load_prediction
# ----------------------------------------------------------
if "load_prediction" not in globals():
    def load_prediction(case_num, out_dir):
        """
        Load the prediction JSON for a single case.

        Args:
            case_num (int): Numeric case identifier (1-500).
            out_dir (str): Version output directory.

        Returns:
            list[dict]: Each dict has 'tooth' and 'surface' keys.
        """
        pred_path = Path(out_dir) / f"case_{case_num}" / f"case_{case_num}.json"
        if not pred_path.exists():
            return []
        with open(pred_path, "r") as f:
            data = json.load(f)
        preds = []
        for t in data.get("teeth_data", []):
            tooth = str(t.get("tooth_id", "Unknown"))
            surface = t.get("predicted_surface_fine", t.get("caries_position_detail", "Other"))
            if surface not in VALID_SURFACES:
                surface = "Other"
            preds.append({"tooth": tooth, "surface": surface})
        return preds

# ----------------------------------------------------------
# Fallback: match_case
# ----------------------------------------------------------
if "match_case" not in globals():
    def match_case(ground_truth, predictions):
        """
        Match ground-truth and predicted surfaces by tooth ID for one case.

        Args:
            ground_truth (list[dict]): GT entries with 'tooth' and 'surface'.
            predictions (list[dict]): Prediction entries with 'tooth' and 'surface'.

        Returns:
            tuple: (y_true, y_pred) lists of matched surface labels.
        """
        pred_dict = {p["tooth"]: p["surface"] for p in predictions}
        y_true = []
        y_pred = []
        for g in ground_truth:
            tooth = g["tooth"]
            gt_surface = g["surface"]
            pred_surface = pred_dict.get(tooth, "Other")
            y_true.append(gt_surface)
            y_pred.append(pred_surface)
        return y_true, y_pred

# ----------------------------------------------------------
# Fallback: evaluate_version
# ----------------------------------------------------------
if "evaluate_version" not in globals():
    def evaluate_version(version):
        """
        Evaluate predictions for a given version against XML ground truth.

        Args:
            version (str): Version tag (e.g. 'Run3').

        Returns:
            tuple: (all_y_true, all_y_pred, f1_macro).
        """
        out_dir = f"PCA_Output_{version}"
        base_gt = RUN3_GT_ROOT
        all_y_true = []
        all_y_pred = []
        for case_num in range(1, 501):
            gt_folder = base_gt / f"case {case_num}"
            ground_truth = parse_case_ground_truth(gt_folder)
            predictions = load_prediction(case_num, out_dir)
            if len(ground_truth) == 0 and len(predictions) == 0:
                continue
            yt, yp = match_case(ground_truth, predictions)
            all_y_true.extend(yt)
            all_y_pred.extend(yp)
        accuracy = accuracy_score(all_y_true, all_y_pred)
        precision = precision_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        recall = recall_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        f1 = f1_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        cm = confusion_matrix(all_y_true, all_y_pred, labels=VALID_SURFACES)
        cm_df = pd.DataFrame(cm, index=VALID_SURFACES, columns=VALID_SURFACES)
        print(f"\n========== FINAL EVALUATION [{version}] ==========")
        print(f"Total Samples : {len(all_y_true)}")
        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Precision     : {precision:.4f}")
        print(f"Recall        : {recall:.4f}")
        print(f"F1 Score      : {f1:.4f}")
        print("\nConfusion Matrix:")
        print(cm_df)
        print("\nClassification Report:")
        print(classification_report(
            all_y_true,
            all_y_pred,
            labels=VALID_SURFACES,
            zero_division=0
        ))
        return all_y_true, all_y_pred, f1

# ==========================================
# 2. File loading and feature extraction
# ==========================================

def _load_json_file(path):
    """
    Load and parse a JSON file from the given path.

    Args:
        path (Path): Filesystem path to the JSON file.

    Returns:
        dict or None: Parsed JSON, or None if the file does not exist.
    """
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def _load_run3_seg_case(case_id):
    """
    Load the tooth segmentation results JSON for a single case.

    Args:
        case_id (int): Numeric case identifier (1-500).

    Returns:
        dict or None: Parsed segmentation data, or None if missing.
    """
    path = RUN3_SEG_ROOT / f'case {case_id}' / f'case_{case_id}_results.json'
    return _load_json_file(path)

def _load_run3_caries_case(case_id):
    """
    Load the caries-to-tooth mapping JSON for a single case.

    Args:
        case_id (int): Numeric case identifier (1-500).

    Returns:
        dict or None: Parsed caries mapping data, or None if missing.
    """
    path = RUN3_CARIES_ROOT / f'case {case_id}' / f'case_{case_id}_caries_mapping.json'
    return _load_json_file(path)

def _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts):
    """
    Extract a dictionary of 13 geometric features for one caries-tooth pair.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        dict or None: Dictionary with 13 feature values, or None if invalid.
    """
    # Remove small noise clusters from caries points.
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return None

    # PCA-align the tooth and caries points.
    center, angle, _ = perform_pca(tooth_pts, tooth_id)
    tooth_rot = rotate(tooth_pts, center, angle)
    caries_rot = rotate(caries_clean, center, angle)

    # Compute the tooth bounding box after rotation.
    bbox_x, bbox_y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return None

    # Normalize caries coordinates relative to tooth bounding box.
    x_rel = np.clip((caries_rot[:, 0] - bbox_x) / w, 0.0, 1.0)
    y_rel = np.clip((caries_rot[:, 1] - bbox_y) / h, 0.0, 1.0)

    return {
        'is_upper': 1 if int(str(tooth_id)[0]) in [1, 2] else 0,
        'x_mean': float(np.mean(x_rel)),
        'y_mean': float(np.mean(y_rel)),
        'x_std': float(np.std(x_rel)),
        'y_std': float(np.std(y_rel)),
        'x_min': float(np.min(x_rel)),
        'x_max': float(np.max(x_rel)),
        'y_min': float(np.min(y_rel)),
        'x_range': float(np.max(x_rel) - np.min(x_rel)),
        'y_range': float(np.max(y_rel) - np.min(y_rel)),
        'x_centroid_dist': float(abs(np.mean(x_rel) - 0.5)),
        'aspect_ratio': float(w / h),
        'coverage': float(len(caries_clean) / (len(tooth_pts) + 1e-6)),
    }

def create_ml_dataset(case_ids):
    """
    Build a labelled ML dataset by extracting features from all cases.

    Args:
        case_ids (list[int]): List of case identifiers to process.

    Returns:
        pd.DataFrame: Labelled dataset with columns
            ['case_id', 'tooth_id', *FEATURE_COLS, 'label'].
    """
    dataset_rows = []
    total_case_count = len(case_ids)
    print(
        f"[RUNNING] Step 1: \u0e40\u0e23\u0e34\u0e48\u0e21\u0e2a\u0e01\u0e31\u0e14 Features \u0e08\u0e32\u0e01\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25 {total_case_count} \u0e40\u0e04\u0e2a...",
        flush=True,
    )

    for i, case_id in enumerate(case_ids):
        # Update progress bar on every iteration.
        _progress_bar(i + 1, total_case_count, "Step 1: \u0e2a\u0e01\u0e31\u0e14 Features")

        seg_data = _load_run3_seg_case(case_id)
        caries_data = _load_run3_caries_case(case_id)
        gt_folder = RUN3_GT_ROOT / f'case {case_id}'

        if seg_data is None or caries_data is None or not gt_folder.exists():
            continue

        ground_truth_list = parse_case_ground_truth(gt_folder)
        ground_truth_lookup = {str(item['tooth']): item['surface'] for item in ground_truth_list}
        if not ground_truth_lookup:
            continue

        segmentation_map = build_seg_map(seg_data)
        for tooth in caries_data.get('teeth_caries_data', []):
            tooth_id = str(tooth.get('tooth_id', ''))
            if tooth_id not in ground_truth_lookup:
                continue

            tooth_pts = segmentation_map.get(tooth_id, [])
            caries_pts = tooth.get('caries_coordinates', [])
            if len(caries_pts) == 0 or len(tooth_pts) < 10:
                continue

            features = _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts)
            if features is None:
                continue

            row = {
                'case_id': int(case_id),
                'tooth_id': tooth_id,
                **features,
                'label': ground_truth_lookup[tooth_id],
            }
            dataset_rows.append(row)

    columns = ['case_id', 'tooth_id', *FEATURE_COLS, 'label']
    feature_dataframe = pd.DataFrame(dataset_rows, columns=columns)
    if not feature_dataframe.empty:
        feature_dataframe = feature_dataframe[columns]
    return feature_dataframe

# ==========================================
# 3. ML model training and prediction
# ==========================================

def train_classify_ml(feature_dataframe):
    """
    Train a Random Forest classifier on the extracted feature dataset.

    Args:
        feature_dataframe (pd.DataFrame): Labelled dataset from create_ml_dataset().

    Returns:
        tuple: (model, test_dataframe, feature_cols).
    """
    if feature_dataframe.empty:
        raise ValueError('ML dataset is empty.')

    # Group-aware train/test split by case_id.
    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(feature_dataframe, groups=feature_dataframe['case_id']))
    train_dataframe = feature_dataframe.iloc[train_idx].reset_index(drop=True)
    test_dataframe = feature_dataframe.iloc[test_idx].reset_index(drop=True)

    # Train Random Forest with balanced class weights.
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=200,
        random_state=42,
    )
    model.fit(train_dataframe[FEATURE_COLS], train_dataframe['label'])

    # Persist model to globals and disk.
    global rf_model, rf_feature_cols
    rf_model = model
    rf_feature_cols = FEATURE_COLS

    joblib.dump(rf_model, 'rf_classify_ml.pkl')
    print('Saved model to rf_classify_ml.pkl')
    return model, test_dataframe, FEATURE_COLS

def classify_ml(tooth_id, tooth_pts, caries_pts):
    """
    Classify a caries lesion surface using the trained Random Forest model.

    Args:
        tooth_id (str): FDI tooth identifier.
        tooth_pts (list): Pixel coordinates of the tooth mask.
        caries_pts (list): Pixel coordinates of the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle, metadata_dict).
    """
    try:
        if rf_model is None:
            return 'Other', 0.0, {}

        features = _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts)
        if features is None:
            return 'Other', 0.0, {}

        feature_values = [features[col] for col in FEATURE_COLS]
        prediction_input_df = pd.DataFrame([feature_values], columns=FEATURE_COLS)
        prediction = rf_model.predict(prediction_input_df)[0]

        return prediction, 0.0, {"method": "RandomForest"}
    except Exception as e:
        print(e)
        return 'Other', 0.0, {}

def process_case_ml(case_id, output_root):
    """
    Run the ML classifier on all caries-affected teeth in one case and
    save predictions as a JSON file.

    Args:
        case_id (int): Numeric case identifier (1-500).
        output_root (Path): Root directory for writing prediction JSONs.

    Returns:
        tuple: (is_success, status_message).
    """
    seg_data = _load_run3_seg_case(case_id)
    caries_data = _load_run3_caries_case(case_id)

    case_dir = output_root / f'case_{case_id}'
    case_dir.mkdir(parents=True, exist_ok=True)

    result = {'case_number': int(case_id), 'teeth_data': []}

    if seg_data is None or caries_data is None:
        with open(case_dir / f'case_{case_id}.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)
        return False, 'Missing input data'

    segmentation_map = build_seg_map(seg_data)
    teeth_caries_list = caries_data.get('teeth_caries_data', [])

    for tooth in teeth_caries_list:
        tooth_id = str(tooth.get('tooth_id', ''))
        tooth_pts = segmentation_map.get(tooth_id, [])
        caries_pts = tooth.get('caries_coordinates', [])

        surface, angle, metadata = classify_ml(tooth_id, tooth_pts, caries_pts)

        result['teeth_data'].append({
            'tooth_id': tooth_id,
            'version': 'Run3',
            'has_caries': True,
            'confidence': float(tooth.get('confidence', 0.0)),
            'caries_position_detail': surface,
            'predicted_surface_fine': surface,
            'tooth_coordinates': tooth_pts,
            'caries_coordinates': caries_pts,
        })

    with open(case_dir / f'case_{case_id}.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)

    return True, f"OK ({len(result['teeth_data'])} teeth)"

def run_classify_ml_pipeline(case_ids):
    """
    Run the complete Run 3 ML pipeline: extract features, train model,
    and predict surfaces for all cases.

    Args:
        case_ids (list[int]): List of case identifiers to process.

    Returns:
        tuple: (model, test_dataframe).
    """
    # --- Step 1: Extract features ---
    print("[START] \u0e40\u0e23\u0e34\u0e48\u0e21\u0e23\u0e31\u0e19 Pipeline Run 3...", flush=True)
    feature_dataframe = create_ml_dataset(case_ids)
    print(
        f'[DONE] Step 1 \u0e40\u0e2a\u0e23\u0e47\u0e08\u0e2a\u0e34\u0e49\u0e19! \u0e44\u0e14\u0e49\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25\u0e40\u0e15\u0e23\u0e35\u0e22\u0e21\u0e40\u0e17\u0e23\u0e19\u0e17\u0e31\u0e49\u0e07\u0e2b\u0e21\u0e14: '
        f'{len(feature_dataframe)} \u0e0b\u0e35\u0e48',
        flush=True,
    )

    # --- Step 2: Train model ---
    print("[RUNNING] Step 2: \u0e01\u0e33\u0e25\u0e31\u0e07 Train \u0e42\u0e21\u0e40\u0e14\u0e25 Random Forest...", flush=True)
    model, test_dataframe, feature_cols = train_classify_ml(feature_dataframe)
    print(
        f'[DONE] Step 2 \u0e40\u0e2a\u0e23\u0e47\u0e08\u0e2a\u0e34\u0e49\u0e19! Train: '
        f'{len(feature_dataframe) - len(test_dataframe)} \u0e0b\u0e35\u0e48 | '
        f'Test: {len(test_dataframe)} \u0e0b\u0e35\u0e48',
        flush=True,
    )

    # --- Step 3: Predict on all 500 cases ---
    success_count = 0
    failure_count = 0
    total_case_count = len(case_ids)

    print("[RUNNING] Step 3: \u0e19\u0e33\u0e42\u0e21\u0e40\u0e14\u0e25\u0e44\u0e1b\u0e17\u0e33\u0e19\u0e32\u0e22\u0e1c\u0e25\u0e17\u0e31\u0e49\u0e07 500 \u0e40\u0e04\u0e2a...", flush=True)
    for i, case_id in enumerate(case_ids):
        is_success, status_message = process_case_ml(case_id, RUN3_OUTPUT_ROOT)
        if is_success:
            success_count += 1
        else:
            failure_count += 1

        # Update the progress bar on every iteration.
        _progress_bar(i + 1, total_case_count, "Step 3: \u0e17\u0e33\u0e19\u0e32\u0e22\u0e1c\u0e25")

    print(
        f'[SUCCESS] \u0e2a\u0e33\u0e40\u0e23\u0e47\u0e08! \u0e40\u0e02\u0e35\u0e22\u0e19\u0e44\u0e1f\u0e25\u0e4c\u0e17\u0e33\u0e19\u0e32\u0e22\u0e1c\u0e25\u0e41\u0e25\u0e49\u0e27: '
        f'{success_count} \u0e40\u0e04\u0e2a, \u0e25\u0e49\u0e21\u0e40\u0e2b\u0e25\u0e27: {failure_count} \u0e40\u0e04\u0e2a',
        flush=True,
    )
    return model, test_dataframe

# ==========================================
# 4. Execution block
# ==========================================

# 4.1 Register Run3 in VERSION_CONFIGS.
if "VERSION_CONFIGS" not in globals() or VERSION_CONFIGS is None:
    VERSION_CONFIGS = {}
if "Run3" not in VERSION_CONFIGS:
    VERSION_CONFIGS["Run3"] = {
        "out_dir": "PCA_Output_Run3",
        "classifier": classify_ml,
        "label": "Geometric Feature + ML Classifier",
    }

# 4.2 Run feature extraction, model training, and prediction on 500 cases.
run3_case_ids = list(range(1, 501))
rf_model, test_dataframe = run_classify_ml_pipeline(run3_case_ids)

# 4.3 Evaluate predictions against ground truth.
all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version('Run3')

# RUN 3: Random Forest classify_ml (Smart Fallback)

In [9]:
# ==========================================
# RF Model Status — Quick diagnostic check
# ==========================================
# Prints the current state of the trained Random Forest model
# (classes, number of estimators, number of features) to verify
# that the model was loaded/trained correctly before proceeding.

print("=== RF Model Status ===")
if "rf_model" in globals():
    print(f"rf_model     : {rf_model}")
    if rf_model is not None:
        # Show the class labels the model was trained on.
        print(f"classes_     : {rf_model.classes_}")
        # Show the number of decision trees in the ensemble.
        print(f"n_estimators : {rf_model.n_estimators}")
        # Show the number of input features expected by the model.
        print(f"n_features   : {rf_model.n_features_in_}")
else:
    print("rf_model is not defined in this session.")

In [ ]:
# =========================================================
# RUN 3: Random Forest classify_ml (Smart Fallback)
# =========================================================
# This cell implements the full Run 3 pipeline:
#   1. Load a pre-trained Random Forest model (rf_classify_ml.pkl)
#   2. Classify each caries lesion using predict_proba with Smart Fallback
#   3. Write per-case prediction JSON files to PCA_Output_Run3/
#   4. Evaluate predictions against XML ground truth
#   5. Generate README_run3.html documentation
# =========================================================

import pandas as pd
import warnings
import json
import joblib
from pathlib import Path


# ----------------------------------------------------------
# Progress bar helper (no external dependencies)
# ----------------------------------------------------------
def _progress_bar(current, total, prefix='Progress', bar_length=30):
    """
    Print an inline text-based progress bar that overwrites itself.

    Args:
        current (int): Current step number (1-indexed).
        total (int): Total number of steps.
        prefix (str): Label displayed before the bar.
        bar_length (int): Character width of the bar.
    """
    fraction = current / max(total, 1)
    filled = int(bar_length * fraction)
    bar = chr(9608) * filled + chr(9617) * (bar_length - filled)
    print(
        f'\r   {prefix} [{bar}] {fraction*100:.0f}% ({current}/{total})',
        end='', flush=True,
    )
    if current >= total:
        print()  # newline when complete


# ----------------------------------------------------------
# Global constants and fallback initialisation
# ----------------------------------------------------------

# FEATURE_COLS: ordered list of the 13 geometric features used by the
# Random Forest model.  The order must match the training schema.
if "FEATURE_COLS" not in globals():
    FEATURE_COLS = [
        "is_upper", "x_mean", "y_mean", "x_std", "y_std",
        "x_min", "x_max", "y_min", "x_range", "y_range",
        "x_centroid_dist", "aspect_ratio", "coverage",
    ]

# RUN3_MODEL_PATH: filesystem path to the serialised Random Forest model.
if "RUN3_MODEL_PATH" not in globals():
    RUN3_MODEL_PATH = Path.cwd() / "rf_classify_ml.pkl"

# rf_model: the loaded sklearn RandomForestClassifier instance.
# Initialised to None and populated at load time below.
if "rf_model" not in globals():
    rf_model = None

# RUN3_OUTPUT_ROOT: directory where per-case prediction JSONs are written.
if "RUN3_OUTPUT_ROOT" not in globals():
    RUN3_OUTPUT_ROOT = Path.cwd() / "PCA_Output_Run3"
    RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# RUN3_SEG_ROOT / RUN3_CARIES_ROOT: input data directories for tooth
# segmentation results and caries-to-tooth mapping results respectively.
if "RUN3_SEG_ROOT" not in globals() or "RUN3_CARIES_ROOT" not in globals():
    _SP_DIR = Path.cwd().parent
    RUN3_SEG_ROOT = _SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition"
    RUN3_CARIES_ROOT = _SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output"

# ----------------------------------------------------------
# 0. Load the pre-trained Random Forest model
# ----------------------------------------------------------
try:
    if RUN3_MODEL_PATH.exists():
        rf_model = joblib.load(RUN3_MODEL_PATH)
        print("[DONE] \u0e42\u0e2b\u0e25\u0e14\u0e42\u0e21\u0e40\u0e14\u0e25 rf_classify_ml.pkl \u0e2a\u0e33\u0e40\u0e23\u0e47\u0e08!", flush=True)
    else:
        print(f"[ERROR] \u0e44\u0e21\u0e48\u0e1e\u0e1a\u0e42\u0e21\u0e40\u0e14\u0e25: {RUN3_MODEL_PATH}", flush=True)
except Exception as e:
    print(f"[ERROR] \u0e42\u0e2b\u0e25\u0e14\u0e42\u0e21\u0e40\u0e14\u0e25\u0e44\u0e21\u0e48\u0e2a\u0e33\u0e40\u0e23\u0e47\u0e08: {e}", flush=True)
    rf_model = None

# ----------------------------------------------------------
# Fallback: _extract_ml_feature_dict
# Provides a no-op stub when the training cell has not been executed.
# ----------------------------------------------------------
if "_extract_ml_feature_dict" not in globals():
    def _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts):
        """
        Stub fallback that returns None (no features extracted).

        Args:
            tooth_id (str): FDI tooth identifier.
            tooth_pts (list): Pixel coordinates of the tooth mask.
            caries_pts (list): Pixel coordinates of the caries region.

        Returns:
            None: Always returns None in the fallback version.
        """
        return None

# ----------------------------------------------------------
# Fallback: classify_xthird (Baseline v4.5 X-Thirds classifier)
# Used as the Smart Fallback target when the RF model cannot predict.
# ----------------------------------------------------------
if "classify_xthird" not in globals():
    if "classify_surface_v45" in globals():
        classify_xthird = classify_surface_v45
    else:
        def classify_xthird(tooth_id, tooth_pts, caries_pts):
            """
            Ultimate fallback classifier when no other method is available.

            Args:
                tooth_id (str): FDI tooth identifier.
                tooth_pts (list): Pixel coordinates of the tooth mask.
                caries_pts (list): Pixel coordinates of the caries region.

            Returns:
                tuple: ("Other", 0.0, {"method": "Fallback-Other"})
            """
            return "Other", 0.0, {"method": "Fallback-Other"}

# ==========================================
# 1. classify_ml \u2014 Smart Fallback with probability filtering
# ==========================================
def classify_ml(tooth_id, tooth_pts, caries_pts):
    """
    Classify a caries lesion surface using the Random Forest model with
    probability-based prediction and Smart Fallback.

    The function first attempts RF prediction via predict_proba.  If the
    model is unavailable or feature extraction fails, it falls back to the
    Baseline X-Thirds classifier (classify_xthird).  Predictions of 'Other'
    are filtered out by considering only Occlusal/Mesial/Distal probabilities.

    Args:
        tooth_id (str): FDI tooth identifier (e.g. "16", "36").
        tooth_pts (list): List of [x, y] pixel coordinates for the tooth mask.
        caries_pts (list): List of [x, y] pixel coordinates for the caries region.

    Returns:
        tuple: (predicted_surface, rotation_angle, metadata_dict)
            - predicted_surface (str): One of "Occlusal", "Mesial", "Distal", "Other".
            - rotation_angle (float): PCA rotation angle in degrees (0.0 for RF path).
            - metadata_dict (dict): Contains the prediction method used.
    """
    try:
        # Extract the 13 geometric features from raw pixel coordinates.
        features = _extract_ml_feature_dict(tooth_id, tooth_pts, caries_pts)

        # Fallback to Baseline if features could not be computed or model is missing.
        if features is None or rf_model is None:
            return classify_xthird(tooth_id, tooth_pts, caries_pts)

        # Build a single-row DataFrame matching the training feature schema.
        prediction_input_df = pd.DataFrame(
            [[features[col] for col in FEATURE_COLS]],
            columns=FEATURE_COLS
        )

        # Predict class probabilities (suppress sklearn warnings).
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            class_probabilities = rf_model.predict_proba(prediction_input_df)[0]

        # Map class indices to class names from the trained model.
        model_classes = list(rf_model.classes_)

        # Filter to only the three valid anatomical surfaces.
        # This prevents the model from predicting 'Other' when trained classes include it.
        valid_surface_classes = ["Occlusal", "Mesial", "Distal"]
        surface_scores = {
            cls: class_probabilities[model_classes.index(cls)]
            for cls in valid_surface_classes
            if cls in model_classes
        }

        # Fallback if none of the valid surfaces are in the model's class list.
        if not surface_scores:
            return classify_xthird(tooth_id, tooth_pts, caries_pts)

        # Select the surface with the highest probability.
        prediction = max(surface_scores, key=surface_scores.get)

        return prediction, 0.0, {"method": "RandomForest_Proba"}
    except Exception:
        # On any unexpected error, fall back to the Baseline classifier.
        try:
            return classify_xthird(tooth_id, tooth_pts, caries_pts)
        except Exception:
            return "Other", 0.0, {}

# ==========================================
# 2. Fallback helper functions (standalone safety)
# ==========================================

# Fallback: _load_json_file \u2014 generic JSON loader with existence check.
if "_load_json_file" not in globals():
    def _load_json_file(path):
        """
        Load and parse a JSON file from the given path.

        Args:
            path (Path): Filesystem path to the JSON file.

        Returns:
            dict or None: Parsed JSON as a dictionary, or None if the file
                does not exist.
        """
        if not path.exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

# Fallback: _load_run3_seg_case \u2014 loads tooth segmentation JSON for one case.
if "_load_run3_seg_case" not in globals():
    def _load_run3_seg_case(case_id):
        """
        Load the tooth segmentation results JSON for a single case.

        Args:
            case_id (int): Numeric case identifier (1-500).

        Returns:
            dict or None: Parsed segmentation data, or None if file is missing.
        """
        path = RUN3_SEG_ROOT / f"case {case_id}" / f"case_{case_id}_results.json"
        return _load_json_file(path)

# Fallback: _load_run3_caries_case \u2014 loads caries mapping JSON for one case.
if "_load_run3_caries_case" not in globals():
    def _load_run3_caries_case(case_id):
        """
        Load the caries-to-tooth mapping JSON for a single case.

        Args:
            case_id (int): Numeric case identifier (1-500).

        Returns:
            dict or None: Parsed caries mapping data, or None if file is missing.
        """
        path = RUN3_CARIES_ROOT / f"case {case_id}" / f"case_{case_id}_caries_mapping.json"
        return _load_json_file(path)

# Fallback: build_seg_map \u2014 builds a tooth_id -> pixel_coordinates lookup.
if "build_seg_map" not in globals():
    def build_seg_map(seg_data):
        """
        Build a mapping from tooth ID to its pixel coordinates.

        Args:
            seg_data (dict): Parsed segmentation JSON containing "teeth_data".

        Returns:
            dict: Mapping of str(tooth_id) -> list of [x, y] coordinates.
        """
        return {
            str(t["tooth_id"]): t.get("pixel_coordinates", [])
            for t in seg_data.get("teeth_data", [])
        }

# Fallback: process_case_ml \u2014 runs classify_ml on every tooth in one case
# and writes the prediction results to a JSON file.
if "process_case_ml" not in globals():
    def process_case_ml(case_id, output_root):
        """
        Run the ML classifier on all caries-affected teeth in one case and
        save predictions as a JSON file.

        Args:
            case_id (int): Numeric case identifier (1-500).
            output_root (Path): Root directory for writing prediction JSONs.

        Returns:
            tuple: (is_success, status_message)
                - is_success (bool): True if prediction completed normally.
                - status_message (str): Human-readable result description.
        """
        # --- Load stage: read segmentation and caries data ---
        seg_data = _load_run3_seg_case(case_id)
        caries_data = _load_run3_caries_case(case_id)

        case_dir = output_root / f"case_{case_id}"
        case_dir.mkdir(parents=True, exist_ok=True)

        result = {"case_number": int(case_id), "teeth_data": []}

        # Write an empty result if input data is missing.
        if seg_data is None or caries_data is None:
            with open(case_dir / f"case_{case_id}.json", "w", encoding="utf-8") as f:
                json.dump(result, f, indent=2)
            return False, "Missing input data"

        # --- Predict stage: classify every caries-affected tooth ---
        segmentation_map = build_seg_map(seg_data)
        teeth_caries_list = caries_data.get("teeth_caries_data", [])

        for tooth in teeth_caries_list:
            tooth_id = str(tooth.get("tooth_id", ""))
            tooth_pts = segmentation_map.get(tooth_id, [])
            caries_pts = tooth.get("caries_coordinates", [])

            # Run the Smart Fallback classifier.
            surface, angle, metadata = classify_ml(tooth_id, tooth_pts, caries_pts)

            result["teeth_data"].append({
                "tooth_id": tooth_id,
                "version": "Run3",
                "has_caries": True,
                "confidence": float(tooth.get("confidence", 0.0)),
                "caries_position_detail": surface,
                "predicted_surface_fine": surface,
                "tooth_coordinates": tooth_pts,
                "caries_coordinates": caries_pts,
            })

        # --- Save stage: write prediction JSON ---
        with open(case_dir / f"case_{case_id}.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        return True, f"OK ({len(result['teeth_data'])} teeth)"

# ==========================================
# 3. Update VERSION_CONFIGS and run prediction loop
# ==========================================

# Register the Smart Fallback classifier in the shared version config.
if "VERSION_CONFIGS" not in globals() or VERSION_CONFIGS is None:
    VERSION_CONFIGS = {}

VERSION_CONFIGS["Run3"] = {
    "out_dir": "PCA_Output_Run3",
    "classifier": classify_ml,
    "label": "Geometric Feature + ML (Smart Fallback)",
}

# --- Prediction loop: run classify_ml on all 500 cases ---
case_ids = list(range(1, 501))
total_case_count = len(case_ids)
success_count, failure_count = 0, 0

print("[START] \u0e40\u0e23\u0e34\u0e48\u0e21\u0e23\u0e31\u0e19 Step 1: Smart Fallback \u0e17\u0e31\u0e49\u0e07 500 \u0e40\u0e04\u0e2a...", flush=True)
for i, case_id in enumerate(case_ids):
    # Process a single case and track success/failure.
    result = process_case_ml(case_id, RUN3_OUTPUT_ROOT)
    is_success = result[0] if isinstance(result, tuple) else bool(result)
    if is_success:
        success_count += 1
    else:
        failure_count += 1

    # Update the progress bar on every iteration.
    _progress_bar(i + 1, total_case_count, "\u0e17\u0e33\u0e19\u0e32\u0e22\u0e1c\u0e25")

print("[SUCCESS] \u0e17\u0e33\u0e19\u0e32\u0e22\u0e1c\u0e25\u0e40\u0e2a\u0e23\u0e47\u0e08\u0e2a\u0e34\u0e49\u0e19! \u0e01\u0e33\u0e25\u0e31\u0e07\u0e23\u0e31\u0e19 Evaluation...", flush=True)

# --- Evaluation: compute metrics against XML ground truth ---
if "evaluate_version" in globals():
    all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version("Run3")
else:
    print("evaluate_version is not available. Run the evaluation cell to compute metrics.")

# ==========================================
# 4. Generate README_run3.html documentation
# ==========================================
HTML_CONTENT = """\
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dental Caries Surface Classification \u2014 Run 3</title>
<style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
                     Helvetica, Arial, sans-serif, "Apple Color Emoji",
                     "Segoe UI Emoji";
        background: #ffffff;
        color: #1f2937;
        line-height: 1.6;
        padding: 2rem;
        max-width: 960px;
        margin: 0 auto;
    }
    h1 { color: #2563eb; font-size: 1.8rem; margin-bottom: 0.5rem; }
    h2 { color: #2563eb; font-size: 1.35rem; margin-top: 2rem; margin-bottom: 0.75rem;
         border-bottom: 2px solid #e5e7eb; padding-bottom: 0.3rem; }
    h3 { color: #374151; font-size: 1.1rem; margin-top: 1.2rem; margin-bottom: 0.4rem; }
    p, li { margin-bottom: 0.5rem; }
    ul { padding-left: 1.5rem; }
    table {
        border-collapse: collapse;
        width: 100%;
        margin: 1rem 0;
        font-size: 0.9rem;
    }
    th, td {
        border: 1px solid #d1d5db;
        padding: 0.5rem 0.75rem;
        text-align: left;
    }
    th { background: #f3f4f6; font-weight: 600; }
    tr:nth-child(even) { background: #f9fafb; }
    tr.highlight { background: #dbeafe; font-weight: 600; }
    pre, code {
        font-family: "Cascadia Code", "Fira Code", Consolas, monospace;
        font-size: 0.85rem;
    }
    pre {
        background: #f3f4f6;
        padding: 1rem;
        border-radius: 6px;
        overflow-x: auto;
        margin: 1rem 0;
        line-height: 1.45;
    }
    code { background: #f3f4f6; padding: 0.15rem 0.35rem; border-radius: 3px; }
    .badge {
        display: inline-block;
        background: #2563eb;
        color: #fff;
        padding: 0.15rem 0.5rem;
        border-radius: 4px;
        font-size: 0.75rem;
        margin-left: 0.3rem;
        vertical-align: middle;
    }
    footer { margin-top: 3rem; padding-top: 1rem; border-top: 1px solid #e5e7eb;
             font-size: 0.8rem; color: #6b7280; }
</style>
</head>
<body>

<h1>Dental Caries Surface Classification <span class="badge">Run 3</span></h1>

<!-- Section 1: Project Overview -->
<h2>1. Project Overview</h2>
<ul>
    <li><strong>Project Name:</strong> Dental Caries Surface Classification \u2014 Run 3</li>
    <li><strong>Objective:</strong> Classify caries lesion surfaces into <em>Occlusal</em>,
        <em>Mesial</em>, or <em>Distal</em> from dental panoramic X-ray images.</li>
    <li><strong>Approach:</strong> Geometric Feature Extraction (PCA-aligned, 13 features)
        + Random Forest Classifier with Smart Fallback to Baseline X-Thirds.</li>
    <li><strong>Dataset:</strong> 500 annotated dental panoramic X-ray cases
        (1,979 caries-affected teeth with ground-truth surface labels).</li>
</ul>

<!-- Section 2: Results Summary -->
<h2>2. Results Summary</h2>
<table>
    <thead>
        <tr>
            <th>Run</th><th>Technique</th>
            <th>Recall Occlusal</th><th>Recall Mesial</th><th>Recall Distal</th>
            <th>Accuracy</th><th>Notes</th>
        </tr>
    </thead>
    <tbody>
        <tr><td>Baseline</td><td>X-Thirds Hard Partition</td><td>0.20</td><td>0.83</td><td>0.83</td><td>0.70</td><td>Starting point</td></tr>
        <tr><td>Run 1</td><td>Biaxial Threshold</td><td>0.63</td><td>0.71</td><td>0.25</td><td>0.48</td><td>Over-predicted Occlusal</td></tr>
        <tr><td>Run 2</td><td>Fuzzy Distance Weighting</td><td>0.79</td><td>0.46</td><td>0.45</td><td>0.52</td><td>Focal point mismatch</td></tr>
        <tr class="highlight"><td>Run 3</td><td>RF + Smart Fallback</td><td>0.84</td><td>0.80</td><td>0.84</td><td>0.83</td><td>TARGET ACHIEVED</td></tr>
    </tbody>
</table>

<!-- Section 3: Feature List -->
<h2>3. Feature List</h2>
<p>The classifier uses 13 geometric features extracted after PCA alignment of each tooth:</p>
<table>
    <thead><tr><th>Feature</th><th>Description</th><th>Why It Matters</th></tr></thead>
    <tbody>
        <tr><td><code>is_upper</code></td><td>1 if upper jaw, 0 if lower</td><td>Crown direction differs by jaw</td></tr>
        <tr><td><code>x_mean</code></td><td>Mean relative X position</td><td>Left/right surface indicator</td></tr>
        <tr><td><code>y_mean</code></td><td>Mean relative Y position</td><td>Vertical position on tooth</td></tr>
        <tr><td><code>x_std</code></td><td>Spread in X axis</td><td>Width of caries lesion</td></tr>
        <tr><td><code>y_std</code></td><td>Spread in Y axis</td><td>Height of caries lesion</td></tr>
        <tr><td><code>x_min</code></td><td>Leftmost X position</td><td>Proximity to mesial boundary</td></tr>
        <tr><td><code>x_max</code></td><td>Rightmost X position</td><td>Proximity to distal boundary</td></tr>
        <tr><td><code>y_min</code></td><td>Topmost Y position</td><td>Proximity to crown surface</td></tr>
        <tr><td><code>x_range</code></td><td>x_max - x_min</td><td>Horizontal extent of lesion</td></tr>
        <tr><td><code>y_range</code></td><td>y_max - y_min</td><td>Vertical extent of lesion</td></tr>
        <tr><td><code>x_centroid_dist</code></td><td>Distance from center X</td><td>How far off-center the lesion is</td></tr>
        <tr><td><code>aspect_ratio</code></td><td>Tooth width / height</td><td>Tooth shape normalization</td></tr>
        <tr><td><code>coverage</code></td><td>Lesion size / tooth size</td><td>Relative lesion severity</td></tr>
    </tbody>
</table>

<!-- Section 4: Pipeline Architecture -->
<h2>4. Pipeline Architecture</h2>
<pre>
XML Ground Truth
     |
     v
parse_case_ground_truth()
     |
create_ml_dataset()  &lt;-- seg JSON + caries JSON + GT XML
     |
_extract_ml_feature_dict()  (PCA align -&gt; normalize -&gt; 13 features)
     |
train_classify_ml()  (GroupShuffleSplit -&gt; RandomForest -&gt; save pkl)
     |
classify_ml()  (load features -&gt; predict proba -&gt; Smart Fallback)
     |
process_case_ml()  (write prediction JSON per case)
     |
evaluate_version()  (confusion matrix + classification report)
</pre>

<!-- Section 5: Key Design Decisions -->
<h2>5. Key Design Decisions</h2>
<h3>5.1 GroupShuffleSplit by case_id</h3>
<p>The dataset is split at the <strong>case level</strong>, not the individual tooth level.
    This prevents data leakage: teeth from the same panoramic X-ray share imaging conditions,
    so GroupShuffleSplit ensures all teeth from a given case are in either train or test, never both.</p>
<h3>5.2 class_weight='balanced' in RandomForest</h3>
<p>The class distribution is imbalanced (Distal &gt; Mesial &gt; Occlusal). Setting
    <code>class_weight='balanced'</code> adjusts sample weights inversely proportional to
    class frequencies, improving Occlusal recall from 0.20 to 0.84.</p>
<h3>5.3 Smart Fallback filters out 'Other' predictions</h3>
<p>The Smart Fallback restricts candidates to ["Occlusal", "Mesial", "Distal"] and selects
    the highest-probability surface. If the model or features are unavailable, it falls back
    to the Baseline X-Thirds classifier.</p>
<h3>5.4 coverage uses len(tooth_pts) as denominator</h3>
<p>Using actual tooth mask pixel count (not bounding-box area) accounts for tooth shape.
    Epsilon (1e-6) prevents division by zero for empty masks.</p>

<!-- Section 6: File Structure -->
<h2>6. File Structure</h2>
<pre>
SP/
+-- data/
|   +-- 500 cases with annotation/case N/*.xml   (Ground Truth)
+-- week2-Tooth Detection &amp; Segmentation/
|   +-- 500-segmentation+recognition/case N/case_N_results.json
+-- week3-Caries-to-Tooth Mapping/
|   +-- dental_analysis_output/case N/case_N_caries_mapping.json
+-- phase2-1april/
    +-- PCA_Output_Run3/case_N/case_N.json       (Predictions)
    +-- rf_classify_ml.pkl                        (Trained Model)
    +-- README_run3.html                          (This file)
</pre>

<footer>
    Generated by <strong>Run 3 Pipeline</strong> &mdash;
    Dental Caries Surface Classification Project
</footer>

</body>
</html>
"""

with open("README_run3.html", "w", encoding="utf-8") as f:
    f.write(HTML_CONTENT)
print("[DONE] README_run3.html saved.")

In [ ]:
# =========================================================
# Feature Importance Plot (for presentation / report)
# =========================================================
# Plots the Random Forest feature importances as a horizontal
# bar chart sorted by importance.  Saves to PNG for slides.
# =========================================================

import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------
# 1. Extract feature importances from the trained model
# ----------------------------------------------------------
if rf_model is None:
    print("[ERROR] rf_model is not loaded. Run the training cell first.")
else:
    # Use the 13-feature FEATURE_COLS from the training cell.
    feature_names = FEATURE_COLS
    importances = rf_model.feature_importances_

    # Sort features by importance (ascending for horizontal bar chart).
    sorted_indices = np.argsort(importances)
    sorted_names = [feature_names[i] for i in sorted_indices]
    sorted_importances = importances[sorted_indices]

    # ----------------------------------------------------------
    # 2. Create the horizontal bar chart
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    # Colour gradient: low importance -> light blue, high -> dark blue.
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(sorted_importances)))

    bars = ax.barh(
        range(len(sorted_names)),
        sorted_importances,
        color=colors,
        edgecolor='white',
        linewidth=0.5,
        height=0.7,
    )

    # Add value labels on each bar.
    for bar_obj, val in zip(bars, sorted_importances):
        ax.text(
            bar_obj.get_width() + 0.003,
            bar_obj.get_y() + bar_obj.get_height() / 2,
            f'{val:.3f}',
            va='center',
            fontsize=9,
            fontweight='bold',
            color='#333333',
        )

    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names, fontsize=10)
    ax.set_xlabel('Feature Importance (Gini)', fontsize=11, fontweight='bold')
    ax.set_title(
        'Random Forest Feature Importance \u2014 Run 3 Dental Caries Classification',
        fontsize=13,
        fontweight='bold',
        pad=15,
    )

    # Clean up spines for a presentation-ready look.
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False)
    ax.set_xlim(0, max(sorted_importances) * 1.18)

    plt.tight_layout()

    # ----------------------------------------------------------
    # 3. Save to PNG
    # ----------------------------------------------------------
    fig.savefig('feature_importance_run3.png', dpi=200, bbox_inches='tight', facecolor='white')
    print('[DONE] Saved feature_importance_run3.png')
    plt.show()

## 4. Evaluation

parse XML ground truth + parse prediction refactor mapping + parser 
ทำ evaluation:
Accuracy
Precision
Recall
F1-score
Confusion Matrix

In [12]:
# =========================================================
# FULL EVALUATION + HYPOTHESIS TESTING (SELF-CONTAINED)
# =========================================================
# This cell defines the ground-truth XML parser, prediction loader,
# case-matching logic, and evaluation function.  It then runs
# evaluation for the Baseline and Run 3 versions.
# =========================================================

from pathlib import Path
import json
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ----------------------------------------------------------
# Progress bar helper (no external dependencies)
# ----------------------------------------------------------
def _progress_bar(current, total, prefix='Progress', bar_length=30):
    """
    Print an inline text-based progress bar that overwrites itself.

    Args:
        current (int): Current step number (1-indexed).
        total (int): Total number of steps.
        prefix (str): Label displayed before the bar.
        bar_length (int): Character width of the bar.
    """
    fraction = current / max(total, 1)
    filled = int(bar_length * fraction)
    bar = chr(9608) * filled + chr(9617) * (bar_length - filled)
    print(
        f'\r   {prefix} [{bar}] {fraction*100:.0f}% ({current}/{total})',
        end='', flush=True,
    )
    if current >= total:
        print()  # newline when complete


# =========================================================
# XML PARSER — AIM namespace constants and SNODENT mappings
# =========================================================

# AIM XML namespace URIs used by the annotation schema.
AIM_NS = "gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM"
ISO_NS = "uri:iso.org:21090"
NS = {"aim": AIM_NS, "iso": ISO_NS}

# SNODENT code -> surface name mapping.
# Maps SNODENT ontology codes found in typeCode/@code to surface labels.
SNODENT_SURFACE_MAP = {
    "144414D": "Occlusal",
    "146014D": "Distal",
    "145374D": "Mesial",
    "144474D": "Occlusal",
    "146074D": "Distal",
    "145434D": "Mesial",
}

# Display name -> surface name mapping (fallback when SNODENT code is missing).
DISPLAY_NAME_TO_SURFACE = {
    "Occlusal surface": "Occlusal",
    "Occlusal Surface": "Occlusal",
    "Distal Surface": "Distal",
    "Distal surface": "Distal",
    "Mesial Surface": "Mesial",
    "Mesial surface": "Mesial",
}

# SNODENT tooth code -> FDI number mapping.
# Maps SNODENT ontology codes to FDI two-digit tooth identifiers.
SNODENT_TO_FDI = {
    "161006D": "11", "160842D": "12", "160288D": "13", "161286D": "14",
    "160450D": "15", "160770D": "16", "161204D": "17", "160618D": "18",
    "160194D": "21", "160132D": "22", "160506D": "23", "161340D": "24",
    "160682D": "25", "161074D": "26", "160386D": "27", "160922D": "28",
    "161136D": "31", "160556D": "32", "160068D": "33", "160326D": "34",
    "161248D": "35", "160730D": "36", "161166D": "37", "160580D": "38",
    "160964D": "41", "160350D": "42", "160894D": "43", "160230D": "44",
    "161412D": "45", "160770D": "46", "161102D": "47", "160488D": "48",
}

# VALID_SURFACES: the four surface classes used in evaluation.
VALID_SURFACES = ["Occlusal", "Mesial", "Distal", "Other"]


def _get_display_name(element):
    """
    Extract the displayName value from an ISO-namespaced XML element.

    Args:
        element (xml.etree.ElementTree.Element): Parent element containing
            an iso:displayName child.

    Returns:
        str: The display name value, or empty string if not found.
    """
    dn = element.find("iso:displayName", NS)
    return dn.get("value", "") if dn is not None else ""


def snodent_display_to_fdi(display_name):
    """
    Convert a SNODENT display name (e.g. 'Upper Right First Molar') to an
    FDI two-digit tooth identifier.

    Parses the jaw position (upper/lower) and laterality (left/right) to
    determine the FDI quadrant (1-4), then parses the tooth type to
    determine the position (1-8).

    Args:
        display_name (str): SNODENT descriptive name for the tooth.

    Returns:
        str: FDI identifier (e.g. '16'), or empty string if unparseable.
    """
    dn = display_name.lower()

    # Determine FDI quadrant from jaw + laterality.
    if "upper" in dn and "right" in dn:
        quadrant = 1
    elif "upper" in dn and "left" in dn:
        quadrant = 2
    elif "lower" in dn and "left" in dn:
        quadrant = 3
    elif "lower" in dn and "right" in dn:
        quadrant = 4
    else:
        return ""

    # Determine tooth position from type name.
    if "central incisor" in dn:
        pos = 1
    elif "lateral incisor" in dn:
        pos = 2
    elif "canine" in dn:
        pos = 3
    elif "first premolar" in dn:
        pos = 4
    elif "second premolar" in dn:
        pos = 5
    elif "first molar" in dn:
        pos = 6
    elif "second molar" in dn:
        pos = 7
    elif "third molar" in dn:
        pos = 8
    else:
        return ""

    return f"{quadrant}{pos}"


def parse_aim_xml(xml_path):
    """
    Parse a single AIM XML annotation file to extract tooth and surface.

    Navigates the AIM XML schema to find the ImagingPhysicalEntity and
    its characteristics.  questionIndex=0 identifies the tooth,
    questionIndex=1 identifies the caries surface.

    Args:
        xml_path (str): Filesystem path to the AIM XML file.

    Returns:
        dict or None: {'tooth': str, 'surface': str}, or None if
            parsing fails or the file has no valid annotation.
    """
    try:
        tree = ET.parse(xml_path)
    except Exception:
        return None

    root = tree.getroot()
    anns = root.find("aim:imageAnnotations", NS)
    if anns is None:
        return None

    ann = anns.find("aim:ImageAnnotation", NS)
    if ann is None:
        return None

    tooth = ""
    surface = ""

    phys_coll = ann.find("aim:imagingPhysicalEntityCollection", NS)
    if phys_coll is not None:
        entity = phys_coll.find("aim:ImagingPhysicalEntity", NS)
        if entity is not None:
            char_coll = entity.find(
                "aim:imagingPhysicalEntityCharacteristicCollection", NS
            )
            if char_coll is not None:
                chars = char_coll.findall(
                    "aim:ImagingPhysicalEntityCharacteristic", NS
                )
                for ch in chars:
                    q_idx_el = ch.find("aim:questionIndex", NS)
                    q_idx = q_idx_el.get("value", "") if q_idx_el is not None else ""

                    tc = ch.find("aim:typeCode", NS)
                    if tc is None:
                        continue

                    code = tc.get("code", "")
                    display = _get_display_name(tc)

                    # questionIndex=0: tooth identification.
                    if q_idx == "0":
                        tooth = snodent_display_to_fdi(display)
                        if not tooth:
                            tooth = SNODENT_TO_FDI.get(code, "")

                    # questionIndex=1: surface classification.
                    elif q_idx == "1":
                        surface = SNODENT_SURFACE_MAP.get(code, "")
                        if not surface:
                            surface = DISPLAY_NAME_TO_SURFACE.get(display, "")

    return {"tooth": tooth, "surface": surface}


# =========================================================
# LOAD FUNCTIONS
# =========================================================

def parse_case_ground_truth(case_folder):
    """
    Parse all AIM XML ground-truth files in a case folder.

    Args:
        case_folder (Path or str): Directory containing *.xml annotation files.

    Returns:
        list[dict]: Each dict has 'tooth' (str FDI) and 'surface' (str).
    """
    ground_truth_list = []
    for xml_file in sorted(Path(case_folder).glob("*.xml")):
        parsed = parse_aim_xml(str(xml_file))
        if parsed is None:
            continue

        tooth = str(parsed.get("tooth", "Unknown"))
        surface = parsed.get("surface", "Other")
        if surface not in VALID_SURFACES:
            surface = "Other"

        ground_truth_list.append({"tooth": tooth, "surface": surface})
    return ground_truth_list


def load_prediction(case_num, out_dir):
    """
    Load the prediction JSON for a single case.

    Args:
        case_num (int): Numeric case identifier (1-500).
        out_dir (str): Version output directory (e.g. 'PCA_Output_Run3').

    Returns:
        list[dict]: Each dict has 'tooth' and 'surface' keys.
    """
    pred_path = Path(out_dir) / f"case_{case_num}" / f"case_{case_num}.json"
    if not pred_path.exists():
        return []

    with open(pred_path, "r") as f:
        data = json.load(f)

    preds = []
    for t in data.get("teeth_data", []):
        tooth = str(t.get("tooth_id", "Unknown"))
        surface = t.get(
            "predicted_surface_fine",
            t.get("caries_position_detail", "Other")
        )
        if surface not in VALID_SURFACES:
            surface = "Other"
        preds.append({"tooth": tooth, "surface": surface})
    return preds


def match_case(ground_truth, predictions):
    """
    Match ground-truth and predicted surfaces by tooth ID for one case.

    Args:
        ground_truth (list[dict]): GT entries with 'tooth' and 'surface'.
        predictions (list[dict]): Prediction entries with 'tooth' and 'surface'.

    Returns:
        tuple: (y_true, y_pred) lists of matched surface labels.
    """
    pred_dict = {p["tooth"]: p["surface"] for p in predictions}
    y_true = []
    y_pred = []
    for g in ground_truth:
        tooth = g["tooth"]
        gt_surface = g["surface"]
        pred_surface = pred_dict.get(tooth, "Other")
        y_true.append(gt_surface)
        y_pred.append(pred_surface)
    return y_true, y_pred


# =========================================================
# EVALUATION FUNCTION
# =========================================================

def evaluate_version(version):
    """
    Evaluate predictions for a given version against XML ground truth.

    Iterates over all 500 cases, matches ground-truth to predictions,
    and computes macro-averaged metrics and a confusion matrix.

    Args:
        version (str): Version tag (e.g. 'Run3', 'Baseline').

    Returns:
        tuple: (all_y_true, all_y_pred, f1_macro)
            - all_y_true (list[str]): Ground-truth surface labels.
            - all_y_pred (list[str]): Predicted surface labels.
            - f1_macro (float): Macro-averaged F1 score.
    """
    out_dir = f"PCA_Output_{version}"
    base_gt = Path("../data/500 cases with annotation")

    all_y_true = []
    all_y_pred = []

    # Iterate over all 500 cases and collect matched labels.
    print(f"[RUNNING] Evaluating {version}...", flush=True)
    for case_num in range(1, 501):
        gt_folder = base_gt / f"case {case_num}"

        ground_truth = parse_case_ground_truth(gt_folder)
        predictions = load_prediction(case_num, out_dir)

        if len(ground_truth) == 0 and len(predictions) == 0:
            continue

        yt, yp = match_case(ground_truth, predictions)
        all_y_true.extend(yt)
        all_y_pred.extend(yp)

        # Update progress bar.
        _progress_bar(case_num, 500, f"Eval {version}")

    # Compute evaluation metrics.
    accuracy = accuracy_score(all_y_true, all_y_pred)
    precision = precision_score(all_y_true, all_y_pred, average="macro", zero_division=0)
    recall = recall_score(all_y_true, all_y_pred, average="macro", zero_division=0)
    f1 = f1_score(all_y_true, all_y_pred, average="macro", zero_division=0)

    # Build confusion matrix as a labelled DataFrame.
    cm = confusion_matrix(all_y_true, all_y_pred, labels=VALID_SURFACES)
    cm_df = pd.DataFrame(cm, index=VALID_SURFACES, columns=VALID_SURFACES)

    # Print the evaluation report.
    print(f"\n========== FINAL EVALUATION [{version}] ==========")
    print(f"Total Samples : {len(all_y_true)}")
    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm_df)

    print("\nClassification Report:")
    print(classification_report(
        all_y_true,
        all_y_pred,
        labels=VALID_SURFACES,
        zero_division=0
    ))

    return all_y_true, all_y_pred, f1


# =========================================================
# RUN EVALUATION
# =========================================================

# Evaluate Baseline (X-Thirds Hard Partition).
all_y_true_45, all_y_pred_45, f1_45 = evaluate_version("Baseline")

# Evaluate Run 3 (Random Forest + Smart Fallback).
all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version("Run3")

# Use Baseline ground truth as the canonical GT reference.
all_y_true = all_y_true_45

# =========================================================
# DEBUG: PRINT + SAVE ALL LISTS
# =========================================================
print("\n========== LIST LENGTH CHECK ==========")
print("all_y_true    :", len(all_y_true))
print("all_y_pred_45 :", len(all_y_pred_45))
print("all_y_pred_Run3 :", len(all_y_pred_Run3))

print("\n========== FIRST 30 SAMPLES ==========")
for i in range(min(30, len(all_y_true))):
    print(
        f"{i+1:04d} | "
        f"GT={all_y_true[i]:10s} | "
        f"v4.5={all_y_pred_45[i]:10s} | "
        f"Run3={all_y_pred_Run3[i]:10s}"
    )

# Save debug text files for offline analysis.
Path("debug_all_y_true.txt").write_text(
    "\n".join(all_y_true),
    encoding="utf-8"
)

Path("debug_all_y_pred_45.txt").write_text(
    "\n".join(all_y_pred_45),
    encoding="utf-8"
)

Path("debug_all_y_pred_Run3.txt").write_text(
    "\n".join(all_y_pred_Run3),
    encoding="utf-8"
)